| Model | What it really is |
|---|---|
| Fraud / AML | Supervised classifier — **only if a binary label exists** (no fabricated targets) |
| Account risk | Heuristic **risk scoring** + Isolation Forest (not supervised) |
| Liquidity | Deterministic **stress score** (not trained) |
| Credit | **Synthetic demo** unless you supply a real credit CSV |
| Anomaly | Isolation Forest `anomaly_score` (never a fraud probability) |

Regulatory reporting is handled by the backend (rules + RAG), not by ML.

## Step 0 — Environment check

In [ ]:
import sys, importlib
for pkg in ["numpy","pandas","sklearn","pydantic","joblib","matplotlib","lightgbm"]:
    try:
        m = importlib.import_module(pkg); print(f"{pkg:12s} {getattr(m,'__version__','?')}")
    except Exception as e:
        print(f"{pkg:12s} MISSING -> {e}")
import pydantic
assert int(pydantic.VERSION.split('.')[0]) >= 2, "Need pydantic>=2:  !pip install -U pydantic"
# lightgbm is optional (falls back to sklearn HistGradientBoosting). To install: !pip install -q lightgbm

## Step 1 — Write the module to disk

In [ ]:
%%writefile risk_sentinel.py
"""
risk_sentinel.py  --  Risk Sentinel ML layer (Banking / NBFC Risk Intelligence Copilot)
=========================================================================================
Colab-safe by design:
  * CSV is streamed in chunks; only the mapped columns are kept, in compact dtypes.
  * All rolling / graph features are fully vectorised (sorted keys + cumulative sums +
    searchsorted). There are NO per-row Python loops and NO pandas rolling-on-groups.
  * Training set is capped (all positives + sampled negatives); evaluation is on full data.
  * Optional lightgbm; falls back to sklearn HistGradientBoosting automatically.

Usage (Colab):
    from risk_sentinel import run_pipeline, Config
    summary = run_pipeline("/content/transactions.csv", Config(out_dir="/content/risk_out"))

NEVER claims more than the data supports:
    fraud/AML classifier  -> only if a binary label column exists
    account risk          -> heuristic RISK SCORING (not a supervised prediction)
    liquidity             -> deterministic STRESS SCORE (not a trained model)
    credit                -> real credit CSV if supplied, else clearly-labelled SYNTHETIC demo
    anomaly               -> unsupervised score, never called fraud probability
"""
from __future__ import annotations

import gc
import hashlib
import json
import os
import re
import time
import warnings
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple, Union

import joblib
import numpy as np
import pandas as pd
from pandas.api.types import union_categoricals
from pydantic import BaseModel, ConfigDict, Field, create_model, field_validator
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_auc_score)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

try:  # LightGBM is preinstalled on Colab; fall back gracefully if missing
    import lightgbm as lgb
    HAVE_LGBM = True
except Exception:  # pragma: no cover
    lgb = None
    HAVE_LGBM = False

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*eval_set.*")
SCHEMA_VERSION = "1.0.0"

# ======================================================================================
# 0. CONFIG
# ======================================================================================


@dataclass
class Config:
    out_dir: str = "risk_sentinel_out"
    chunksize: int = 500_000
    max_rows: int = 4_000_000          # safety cap for Colab RAM (first N rows of the file)
    train_frac: float = 0.60           # by ROW QUANTILE OF TIME, not random
    val_frac: float = 0.20
    max_train_rows: int = 1_500_000    # all positives kept; negatives down-sampled
    max_val_rows: int = 1_000_000
    n_estimators: int = 600
    seed: int = 42
    numeric_time_scale: Optional[float] = None   # seconds per unit if timestamp is numeric
    credit_csv: Optional[str] = None             # optional REAL credit dataset
    credit_n_synthetic: int = 20_000
    top_k: Tuple[int, ...] = (50, 100, 500)
    verbose: bool = True


def _log(cfg_or_flag, *a):
    on = cfg_or_flag.verbose if isinstance(cfg_or_flag, Config) else bool(cfg_or_flag)
    if on:
        print(*a, flush=True)


def _show(title: str, obj: Any):
    print(f"\n=== {title} ===")
    try:
        from IPython.display import display  # type: ignore
        display(obj)
    except Exception:
        print(obj.to_string() if hasattr(obj, "to_string") else obj)


# ======================================================================================
# 1. PYDANTIC SCHEMAS  (input -> typed features -> output)
# ======================================================================================

RiskLevel = Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"]
RiskType = Literal["TRANSACTION_FRAUD", "TRANSACTION_AML", "ACCOUNT_AML", "LIQUIDITY", "CREDIT", "ANOMALY"]
ScoringMethod = Literal["supervised_classifier", "risk_scoring_heuristic", "deterministic_stress_score",
                        "unsupervised_anomaly", "synthetic_demo_model"]


class TransactionInput(BaseModel):
    """The ONLY shape the transaction models accept. Arbitrary raw JSON is rejected."""
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    transaction_id: str = Field(min_length=1)
    sender_id: str = Field(min_length=1)
    receiver_id: str = Field(min_length=1)
    amount: float = Field(gt=0, allow_inf_nan=False)
    timestamp: datetime
    transaction_type: Optional[str] = None
    currency: Optional[str] = None

    @field_validator("transaction_id", "sender_id", "receiver_id", mode="before")
    @classmethod
    def _ids_to_str(cls, v):
        if v is None or isinstance(v, bool):
            raise ValueError("id must be a non-empty string/int")
        return str(v)

    @field_validator("timestamp")
    @classmethod
    def _utc_naive(cls, v: datetime):
        if v.tzinfo is not None:
            v = v.astimezone(timezone.utc).replace(tzinfo=None)
        if v.year < 1990:
            raise ValueError("timestamp before 1990 is not plausible")
        return v


class CreditInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    borrower_id: str = Field(min_length=1)
    loan_amount: float = Field(gt=0, allow_inf_nan=False)
    outstanding_amount: float = Field(ge=0, allow_inf_nan=False)
    annual_income: float = Field(gt=0, allow_inf_nan=False)
    monthly_debt_payment: float = Field(ge=0, allow_inf_nan=False)
    credit_utilization: float = Field(ge=0, le=1.5)
    delinquencies_12m: int = Field(ge=0, le=50)
    on_time_payment_ratio: float = Field(ge=0, le=1)
    tenure_months: int = Field(ge=0, le=600)


class RiskPrediction(BaseModel):
    """Strict output contract returned by every inference function."""
    model_config = ConfigDict(extra="forbid")
    entity_id: str
    risk_type: RiskType
    risk_score: float = Field(ge=0.0, le=1.0)
    risk_level: RiskLevel
    fraud_probability: Optional[float] = Field(default=None, ge=0.0, le=1.0)
    anomaly_score: Optional[float] = Field(default=None, ge=0.0, le=1.0)
    signals: List[str] = Field(default_factory=list)
    feature_summary: Dict[str, Optional[float]] = Field(default_factory=dict)
    model_name: str
    model_version: str
    prediction_timestamp: datetime
    scoring_method: ScoringMethod


# ---- feature definitions (order matters: this is the model's input order) -----------
FEATURE_COLUMNS: List[str] = [
    "amount", "log_amount", "hour", "day_of_week", "is_weekend", "is_night", "type_code", "currency_code", "is_self_transfer",
    "s_cnt_5m", "s_cnt_1h", "s_cnt_24h", "s_amt_1h", "s_amt_24h",
    "s_hist_cnt", "s_hist_mean_log", "s_hist_std_log", "s_amt_ratio", "s_amt_z", "s_secs_since_last",
    "r_hist_cnt", "r_hist_mean_log", "r_amt_ratio", "r_in_cnt_1h", "r_in_cnt_24h",
    "s_out_deg", "s_in_deg", "r_in_deg", "r_out_deg", "s_fanout_ratio", "r_fanin_ratio",
    "is_new_pair", "pair_hist_cnt", "pair_share", "reverse_pair_cnt",
    "s_inflow_1h", "s_inflow_24h", "pass_through_24h", "rapid_move_flag",
]
TransactionFeatures = create_model(  # typed feature schema (floats; NaN = "not enough history")
    "TransactionFeatures", __config__=ConfigDict(extra="forbid"),
    **{c: (Optional[float], None) for c in FEATURE_COLUMNS})

ANOMALY_FEATURES = ["log_amount", "s_cnt_1h", "s_cnt_24h", "s_amt_24h", "s_amt_z", "s_hist_cnt", "r_hist_cnt",
                    "r_in_cnt_24h", "s_out_deg", "r_in_deg", "pair_hist_cnt", "s_inflow_24h",
                    "pass_through_24h", "s_secs_since_last", "is_new_pair"]

SIGNAL_TEXT = {  # feature -> human-readable signal when value >= train 99th percentile
    "s_cnt_1h": "sender_velocity_1h_extreme",
    "s_cnt_24h": "sender_velocity_24h_extreme",
    "s_amt_24h": "sender_24h_outflow_extreme",
    "s_amt_z": "amount_far_above_sender_norm",
    "r_in_cnt_1h": "receiver_fan_in_burst_1h",
    "r_in_cnt_24h": "receiver_fan_in_burst_24h",
    "r_in_deg": "receiver_many_distinct_senders",
    "s_out_deg": "sender_many_distinct_receivers",
    "pass_through_24h": "amount_mirrors_recent_inflow",
    "s_inflow_1h": "funds_received_within_last_hour",
}

# ======================================================================================
# 2. HELPERS: time, metrics, serialization
# ======================================================================================


def _epoch_seconds(dt: pd.Series) -> np.ndarray:
    dt = pd.to_datetime(dt, utc=True, errors="coerce")
    return ((dt - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(seconds=1)).to_numpy()


def _ts_str(sec: float) -> str:
    return datetime.fromtimestamp(float(sec), tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def _clean(o: Any):
    """Make anything JSON-safe (numpy -> python, NaN -> None)."""
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (datetime, pd.Timestamp)):
        return str(o)
    return o


def _slog1p(x):
    x = np.asarray(x, dtype=np.float64)
    return np.sign(x) * np.log1p(np.abs(x))


def eval_scores(y, p, ks=(50, 100, 500), thr: Optional[float] = None) -> Dict[str, Any]:
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    n_pos = int(y.sum())
    out: Dict[str, Any] = {"n": int(len(y)), "n_pos": n_pos, "base_rate": float(y.mean()) if len(y) else None}
    if 0 < n_pos < len(y):
        out["pr_auc"] = float(average_precision_score(y, p))
        out["lift_vs_random"] = out["pr_auc"] / out["base_rate"]
        out["roc_auc"] = float(roc_auc_score(y, p))
    else:
        out["pr_auc"] = out["roc_auc"] = None
    order = np.argsort(-p, kind="stable")
    cum = np.cumsum(y[order])
    for k in ks:
        kk = min(k, len(y))
        out[f"precision@{k}"] = float(cum[kk - 1] / kk) if kk else None
        out[f"recall@{k}"] = float(cum[kk - 1] / n_pos) if (kk and n_pos) else None
    if 0.0 <= p.min() and p.max() <= 1.0 and len(y):
        out["brier"] = float(brier_score_loss(y, p))
    if thr is not None:
        pred = p >= thr
        tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
        fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        out.update(threshold=float(thr), precision=prec, recall=rec, accuracy=(tp + tn) / max(len(y), 1),
                   f1=(2 * prec * rec / (prec + rec)) if prec + rec else 0.0,
                   confusion_matrix={"tn": tn, "fp": fp, "fn": fn, "tp": tp})
    return out


def choose_thresholds(y, p) -> Dict[str, float]:
    """medium / high / critical thresholds from VALIDATION data only."""
    y = np.asarray(y).astype(int); p = np.asarray(p, float)
    if y.sum() >= 5:
        prec, rec, thr = precision_recall_curve(y, p)
        f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
        high = float(thr[int(np.argmax(f1))])
        ok = np.where(prec[:-1] >= 0.5)[0]
        crit = float(thr[ok[0]]) if len(ok) else float(np.quantile(p, 0.999))
        okm = np.where(rec[:-1] >= 0.8)[0]
        med = float(thr[okm[-1]]) if len(okm) else float(np.quantile(p, 0.95))
    else:
        med, high, crit = (float(np.quantile(p, q)) for q in (0.95, 0.99, 0.999))
    med, high, crit = sorted([med, high, crit])
    return {"medium": med, "high": high, "critical": crit}


def level_from_score(score: float, th: Dict[str, float]) -> str:
    if score >= th["critical"]:
        return "CRITICAL"
    if score >= th["high"]:
        return "HIGH"
    if score >= th["medium"]:
        return "MEDIUM"
    return "LOW"


# ======================================================================================
# 3. DATASET LOADING, AUTO COLUMN-MAPPING, INSPECTION (streaming; Colab-safe)
# ======================================================================================

_MAP_SPEC = {  # canonical -> (exact normalised names, substrings)
    "sender_id": (["nameorig", "senderid", "sender", "fromaccount", "from", "sourceaccount", "origaccount",
                   "originator", "payer", "account"], ["sender", "fromacc", "orig", "source", "payer", "debit"]),
    "receiver_id": (["namedest", "receiverid", "receiver", "toaccount", "to", "destinationaccount", "destination",
                     "beneficiary", "payee", "account1"], ["receiver", "toacc", "dest", "benef", "payee", "credit"]),
    "amount": (["amount", "amt", "transactionamount", "txnamount", "amountpaid", "amountreceived", "value"],
               ["amount"]),
    "timestamp": (["timestamp", "datetime", "transactiontime", "transactiondate", "date", "time", "step",
                   "trandate", "txntime", "txndate"], ["timestamp", "datetime", "date", "time"]),
    "label": (["islaundering", "isfraud", "fraud", "isfraudulent", "label", "class", "target", "laundering",
               "fraudflag", "y"], ["launder", "fraud"]),
    "transaction_id": (["transactionid", "txnid", "txid", "transid", "referenceid", "id"], []),
    "transaction_type": (["type", "transactiontype", "txntype", "paymentformat", "paymentchannel", "channel",
                          "paymenttype", "transactionchannel"], ["type", "channel", "format"]),
    "currency": (["currency", "paymentcurrency", "receivingcurrency"], ["currency"]),
}
_NEVER_MATCH = ("flagged", "balance")  # system rule outputs / post-transaction balances are leakage-prone


def _norm(c: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(c).lower())


def auto_map_columns(columns) -> Dict[str, Optional[str]]:
    """Map dataset columns -> canonical roles. Never assumes names; returns None when not found."""
    norm = {c: _norm(c) for c in columns}
    used: set = set()
    mapping: Dict[str, Optional[str]] = {}
    for role in ["receiver_id", "sender_id", "amount", "timestamp", "label", "transaction_id",
                 "transaction_type", "currency"]:
        exact, subs = _MAP_SPEC[role]
        pick = None
        for e in exact:
            for c, n in norm.items():
                if c not in used and n == e and not any(b in n for b in _NEVER_MATCH):
                    pick = c; break
            if pick:
                break
        if pick is None:
            for sname in subs:
                for c, n in norm.items():
                    if c not in used and sname in n and not any(b in n for b in _NEVER_MATCH):
                        pick = c; break
                if pick:
                    break
        mapping[role] = pick
        if pick:
            used.add(pick)
    return mapping


def _to_binary(s: pd.Series) -> Optional[pd.Series]:
    """Return a 0/1 int8 series or None if the column is not a clean binary label."""
    if s.dtype == bool:
        return s.astype("int8")
    num = pd.to_numeric(s, errors="coerce")
    if num.notna().mean() > 0.95:
        vals = set(num.dropna().unique().tolist())
        return num.fillna(0).astype("int8") if vals <= {0, 1, 0.0, 1.0} else None
    m = {"1": 1, "yes": 1, "y": 1, "true": 1, "t": 1, "fraud": 1, "fraudulent": 1, "laundering": 1, "suspicious": 1,
         "0": 0, "no": 0, "n": 0, "false": 0, "f": 0, "legit": 0, "legitimate": 0, "normal": 0, "genuine": 0}
    mapped = s.astype(str).str.strip().str.lower().map(m)
    return mapped.fillna(0).astype("int8") if mapped.notna().mean() > 0.95 else None


def detect_credit_columns(columns) -> Dict[str, List[str]]:
    pats = {"default_label": ["default", "delinquen", "chargeoff", "loanstatus"],
            "borrower_features": ["income", "loanamount", "utilization", "utilisation", "dti", "debttoincome",
                                  "creditscore", "installment", "outstanding", "repayment", "ontime"]}
    out = {k: [] for k in pats}
    for c in columns:
        n = _norm(c)
        for k, ps in pats.items():
            if any(p in n for p in ps):
                out[k].append(c)
    return out


def dataset_fingerprint(path: str, n_rows: int) -> str:
    h = hashlib.md5()
    h.update(str(os.path.getsize(path)).encode())
    with open(path, "rb") as f:
        h.update(f.read(1 << 20))
    h.update(str(n_rows).encode())
    return h.hexdigest()[:12]


def load_dataset(path: str, cfg: Config) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Stream the CSV once: build inspection report + compact canonical frame
    (sender_id, receiver_id, amount, ts[int epoch sec], transaction_type, label)."""
    t0 = time.time()
    head = pd.read_csv(path, nrows=5000)
    cols = list(head.columns)
    mapping = auto_map_columns(cols)
    miss = [k for k in ("sender_id", "receiver_id", "amount", "timestamp") if mapping[k] is None]
    if miss:
        raise ValueError(f"Could not map required columns {miss}. Columns found: {cols}. "
                         f"Rename them or extend _MAP_SPEC. Nothing was assumed.")
    _show("COLUMN MAPPING (auto-detected; verify!)",
          pd.DataFrame({"role": list(mapping), "dataset_column": list(mapping.values())}))

    # --- timestamp mode decided from the first rows ---
    tcol = mapping["timestamp"]
    if pd.api.types.is_numeric_dtype(head[tcol]):
        ts_mode = "numeric"
        if cfg.numeric_time_scale is not None:
            scale, origin = cfg.numeric_time_scale, 0.0
        else:
            mx = float(head[tcol].max())
            if _norm(tcol) == "step":                 scale, origin = 3600.0, 1577836800.0  # hours since 2020-01-01
            elif mx > 1e11:                           scale, origin = 0.001, 0.0           # epoch millis
            elif mx > 1e8:                            scale, origin = 1.0, 0.0             # epoch seconds
            else:                                     scale, origin = 3600.0, 1577836800.0  # assume hours
            print(f"[time] numeric timestamp '{tcol}' -> assuming {scale}s per unit (override via Config.numeric_time_scale)")
    else:
        ts_mode, scale, origin = "datetime", 1.0, 0.0

    num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(head[c])]
    obj_cols = [c for c in cols if c not in num_cols]
    st = {"rows": 0, "nulls": pd.Series(0, index=cols, dtype="int64"),
          "sum": pd.Series(0.0, index=num_cols), "min": pd.Series(np.inf, index=num_cols),
          "max": pd.Series(-np.inf, index=num_cols), "cnt": pd.Series(0, index=num_cols, dtype="int64")}
    samples: List[pd.DataFrame] = []
    uniques: Dict[str, set] = {c: set() for c in obj_cols}
    highcard: set = set()
    row_hashes: List[np.ndarray] = []
    id_hashes: List[np.ndarray] = []
    parts = {k: [] for k in ("sender_id", "receiver_id", "transaction_type", "currency")}
    amt_l, ts_l, lab_l = [], [], []
    label_ok = mapping["label"] is not None
    truncated = False

    for ch in pd.read_csv(path, chunksize=cfg.chunksize):
        if st["rows"] >= cfg.max_rows:
            truncated = True; break
        if st["rows"] + len(ch) > cfg.max_rows:
            ch = ch.iloc[: cfg.max_rows - st["rows"]]; truncated = True
        st["rows"] += len(ch)
        st["nulls"] += ch.isna().sum()
        row_hashes.append(pd.util.hash_pandas_object(ch, index=False).to_numpy())
        if mapping["transaction_id"]:
            id_hashes.append(pd.util.hash_pandas_object(ch[mapping["transaction_id"]], index=False).to_numpy())
        for c in num_cols:
            v = pd.to_numeric(ch[c], errors="coerce")
            st["sum"][c] += float(v.sum()); st["cnt"][c] += int(v.notna().sum())
            if v.notna().any():
                st["min"][c] = min(st["min"][c], float(v.min())); st["max"][c] = max(st["max"][c], float(v.max()))
        samples.append(ch[num_cols].sample(min(len(ch), 20_000), random_state=cfg.seed) if num_cols else ch.iloc[:0, :0])
        for c in obj_cols:
            if c in highcard:
                continue
            uniques[c].update(ch[c].dropna().astype(str).unique()[:60].tolist())
            if len(uniques[c]) > 50:
                highcard.add(c); uniques[c] = set()
        # ---- canonical extraction ----
        parts["sender_id"].append(pd.Categorical(ch[mapping["sender_id"]].astype(str)))
        parts["receiver_id"].append(pd.Categorical(ch[mapping["receiver_id"]].astype(str)))
        parts["transaction_type"].append(pd.Categorical(
            ch[mapping["transaction_type"]].astype(str) if mapping["transaction_type"] else pd.Series("NA", index=ch.index)))
        parts["currency"].append(pd.Categorical(
            ch[mapping["currency"]].astype(str) if mapping["currency"] else pd.Series("NA", index=ch.index)))
        amt_l.append(pd.to_numeric(ch[mapping["amount"]], errors="coerce").to_numpy(np.float64))
        if ts_mode == "datetime":
            ts_l.append(_epoch_seconds(ch[tcol]).astype(np.float64))
        else:
            ts_l.append(origin + pd.to_numeric(ch[tcol], errors="coerce").to_numpy(np.float64) * scale)
        if label_ok:
            b = _to_binary(ch[mapping["label"]])
            if b is None:
                label_ok = False
            else:
                lab_l.append(b.to_numpy())
    if truncated:
        print(f"[WARN] max_rows={cfg.max_rows:,} reached -> only the FIRST {st['rows']:,} rows were used.")

    df = pd.DataFrame({
        "sender_id": pd.Series(union_categoricals(parts["sender_id"])),
        "receiver_id": pd.Series(union_categoricals(parts["receiver_id"])),
        "amount": np.concatenate(amt_l), "ts": np.concatenate(ts_l),
        "transaction_type": pd.Series(union_categoricals(parts["transaction_type"])),
        "currency": pd.Series(union_categoricals(parts["currency"]))})
    if label_ok and len(lab_l) == len(amt_l):
        df["label"] = np.concatenate(lab_l)
    else:
        label_ok = False
        if mapping["label"]:
            print(f"[WARN] column '{mapping['label']}' is not a clean binary label -> NOT used as target.")
    del parts, amt_l, ts_l, lab_l; gc.collect()

    # ---- inspection report ----
    all_h = np.concatenate(row_hashes)
    dup_rows = int(len(all_h) - np.unique(all_h).size)
    uniq_tx = int(np.unique(np.concatenate(id_hashes)).size) if id_hashes else None
    med = pd.concat(samples).median() if num_cols else pd.Series(dtype=float)
    num_tbl = pd.DataFrame({"min": st["min"], "max": st["max"],
                            "mean": st["sum"] / st["cnt"].replace(0, np.nan), "median(approx)": med})
    cols_tbl = pd.DataFrame({"dtype": head.dtypes.astype(str), "missing": st["nulls"],
                             "missing_%": (100 * st["nulls"] / max(st["rows"], 1)).round(3)})
    cat_tbl = {c: (sorted(uniques[c]) if c not in highcard else ">50 distinct (high cardinality)") for c in obj_cols}
    _show("SHAPE", f"rows={st['rows']:,} columns={len(cols)}")
    _show("COLUMNS / DTYPES / MISSING", cols_tbl)
    _show("NUMERIC SUMMARY", num_tbl)
    _show("CATEGORICAL UNIQUE VALUES", pd.Series(cat_tbl, dtype=object))
    n_acc = int(len(set(df["sender_id"].cat.categories) | set(df["receiver_id"].cat.categories)))
    report = {
        "file": str(path), "rows": st["rows"], "columns": cols, "n_columns": len(cols),
        "dtypes": head.dtypes.astype(str).to_dict(), "missing": st["nulls"].to_dict(),
        "duplicate_rows": dup_rows, "unique_transaction_ids": uniq_tx if uniq_tx is not None else "no id column mapped",
        "unique_accounts": n_acc, "mapping": mapping, "truncated": truncated,
        "numeric_summary": num_tbl.to_dict(), "categorical_values": cat_tbl,
        "credit_columns_detected": detect_credit_columns(cols),
        "balance_like_columns_excluded_from_features": [c for c in cols if "balance" in _norm(c)],
        "label_available": bool(label_ok), "label_column": mapping["label"] if label_ok else None,
        "dataset_version": dataset_fingerprint(path, st["rows"]),
    }
    print(f"duplicate rows: {dup_rows:,} | unique accounts: {n_acc:,} | unique transaction ids: {report['unique_transaction_ids']}")

    # ---- cleaning (vectorised) ----
    n0 = len(df)
    bad = df["amount"].isna() | ~np.isfinite(df["amount"]) | (df["amount"] <= 0) | df["ts"].isna()
    df = df.loc[~bad].reset_index(drop=True)
    df["ts"] = df["ts"].astype(np.int64)
    order = np.argsort(df["ts"].to_numpy(), kind="stable")           # chronological order
    df = df.iloc[order].reset_index(drop=True)
    report["rows_dropped_invalid"] = int(n0 - len(df))
    report["timestamp_min"], report["timestamp_max"] = _ts_str(df.ts.iloc[0]), _ts_str(df.ts.iloc[-1])
    print(f"cleaning: dropped {n0 - len(df):,} rows (non-positive / missing amount or timestamp). "
          f"Time range: {report['timestamp_min']} -> {report['timestamp_max']}")
    if label_ok:
        vc = df["label"].value_counts().to_dict()
        report["class_distribution"] = {int(k): int(v) for k, v in vc.items()}
        _show("TARGET CLASS DISTRIBUTION", pd.Series(vc, name=mapping["label"]).to_frame())
    else:
        print("[INFO] No usable binary label found -> supervised fraud/AML model will NOT be trained (no fabricated targets).")

    # ---- Pydantic conformance check on a sample (full per-row validation would be too slow) ----
    samp = df.sample(min(1000, len(df)), random_state=cfg.seed)
    errs = 0
    for i, r in samp.iterrows():
        try:
            TransactionInput(transaction_id=str(i), sender_id=r.sender_id, receiver_id=r.receiver_id,
                             amount=float(r.amount), timestamp=datetime.fromtimestamp(int(r.ts), tz=timezone.utc))
        except Exception:
            errs += 1
    report["pydantic_sample_errors"] = errs
    print(f"pydantic TransactionInput check on {len(samp)} sampled rows: {errs} errors | load time {time.time()-t0:.0f}s")
    return df, report


# ======================================================================================
# 4. FEATURE ENGINEERING  (vectorised, strictly-past, shared by training AND inference)
# ======================================================================================
"""
LEAKAGE REGISTER
 L1  Random splits would mix future into train        -> time-quantile split (train<val<test).
 L2  Rolling/expanding stats using the current row    -> every window query is [t-W, t) STRICTLY before t.
 L3  Same-timestamp rows                              -> excluded from each other's windows (strict <).
 L4  Per-account/global stats fitted on full data     -> only train-period data is used to fit encoders,
                                                          imputers, scalers, quantile references, IF model.
 L5  Post-transaction balances (oldbalance/newbalance)-> excluded from all models (they reveal the outcome).
 L6  System fraud flags (e.g. isFlaggedFraud)         -> never mapped as label or feature.
 L7  Calibration/thresholds fitted on test            -> fitted on VALIDATION only; test used once.
 L8  Degree / unique-counterparty counts              -> cumulative counts of first-seen pairs strictly before t.
 L9  Account-level scorer                             -> reference distributions from data <= end of validation;
                                                          back-tested on the later TEST period.
 L10 Negative down-sampling                           -> TRAIN only; val/test keep natural class balance.
 L11 Median -> expanding MEAN of log-amount is used (exact past-only median is O(n^2)); documented deviation.
"""


class _Events:
    """Event table sorted by (group,time) with cumulative sums -> O(log n) window queries."""

    def __init__(self, g: np.ndarray, t: np.ndarray, vals: List[np.ndarray], big: int):
        order = np.lexsort((t, g))
        self.big = int(big)
        self.key = g[order].astype(np.int64) * self.big + t[order]
        v = np.column_stack(vals)[order].astype(np.float64)
        self.cs = np.vstack([np.zeros((1, v.shape[1])), np.cumsum(v, axis=0)])

    def query(self, qg: np.ndarray, qt: np.ndarray, window: Optional[int] = None):
        qk = qg.astype(np.int64) * self.big + qt
        right = np.searchsorted(self.key, qk, side="left")              # events with time < qt  (strict)
        left = np.searchsorted(self.key, qg.astype(np.int64) * self.big if window is None else qk - window, side="left")
        return (right - left).astype(np.float64), self.cs[right] - self.cs[left]

    def last_gap(self, qg: np.ndarray, qt: np.ndarray) -> np.ndarray:
        qk = qg.astype(np.int64) * self.big + qt
        right = np.searchsorted(self.key, qk, side="left")
        gstart = np.searchsorted(self.key, qg.astype(np.int64) * self.big, side="left")
        has = right > gstart
        prev_t = self.key[np.maximum(right - 1, 0)] - qg.astype(np.int64) * self.big
        return np.where(has, qt - prev_t, np.nan).astype(np.float64)


def _encode_ids(s: pd.Series, r: pd.Series):
    cat = union_categoricals([pd.Categorical(s), pd.Categorical(r)])
    codes = np.asarray(cat.codes, dtype=np.int64)
    n = len(s)
    return codes[:n], codes[n:], cat.categories


def build_type_map(types: pd.Series) -> Dict[str, int]:
    return {str(v): i for i, v in enumerate(sorted(pd.Series(types).astype(str).unique()))}


def build_transaction_features(df: pd.DataFrame, type_map: Optional[Dict[str, int]] = None,
                               currency_map: Optional[Dict[str, int]] = None) -> pd.DataFrame:
    """df columns: sender_id, receiver_id, amount, ts (int epoch seconds), [transaction_type].
    Returns float32 DataFrame[FEATURE_COLUMNS] aligned to df rows. Uses ONLY strictly-earlier data."""
    n = len(df)
    if n == 0:
        return pd.DataFrame(columns=FEATURE_COLUMNS, dtype=np.float32)
    s, r, _ = _encode_ids(df["sender_id"], df["receiver_id"])
    amt = df["amount"].to_numpy(np.float64)
    ts_abs = df["ts"].to_numpy(np.int64)
    ts = ts_abs - ts_abs.min()
    big = int(ts.max()) + 8 * 86400 + 10
    lamt = np.log1p(amt)
    vals = [amt, lamt, lamt ** 2]
    H, D, W5 = 3600, 86400, 300

    E_s = _Events(s, ts, vals, big)          # events keyed by sender account (as sender)
    E_r = _Events(r, ts, vals, big)          # events keyed by receiver account (as receiver)

    c5, a5 = E_s.query(s, ts, W5); c1, a1 = E_s.query(s, ts, H); c24, a24 = E_s.query(s, ts, D)
    sh_c, sh_v = E_s.query(s, ts)
    rh_c, rh_v = E_r.query(r, ts)
    rin1, _ = E_r.query(r, ts, H); rin24, _ = E_r.query(r, ts, D)
    _, sin1 = E_r.query(s, ts, H); _, sin24 = E_r.query(s, ts, D)    # inflow to the SENDER account recently

    # pair / graph structure
    M = int(max(s.max(), r.max())) + 1
    pc = s * M + r
    upc = np.unique(pc)
    pid = np.searchsorted(upc, pc)
    rc = r * M + s
    pos = np.searchsorted(upc, rc)
    found = (pos < len(upc)) & (upc[np.minimum(pos, len(upc) - 1)] == rc)
    rid = np.where(found, pos, -1)
    E_p = _Events(pid, ts, [amt], big)
    pair_c, _ = E_p.query(pid, ts)
    rev_c, _ = E_p.query(rid, ts)
    order = np.argsort(ts, kind="stable")
    _, first_idx = np.unique(pc[order], return_index=True)
    first = np.zeros(n, dtype=np.float64); first[order[first_idx]] = 1.0
    E_out = _Events(s, ts, [first], big)     # new (sender->receiver) pair appearing: out-degree of sender
    E_in = _Events(r, ts, [first], big)      # ... in-degree of receiver
    _, s_out = E_out.query(s, ts); _, s_in = E_in.query(s, ts)
    _, r_in = E_in.query(r, ts);   _, r_out = E_out.query(r, ts)

    def mean_std(cnt, v):
        with np.errstate(all="ignore"):
            mean = np.where(cnt > 0, v[:, 1] / cnt, np.nan)
            var = np.where(cnt > 1, np.maximum(v[:, 2] / cnt - mean ** 2, 0.0), np.nan)
        return mean, np.sqrt(var)

    s_mean, s_std = mean_std(sh_c, sh_v)
    r_mean, _ = mean_std(rh_c, rh_v)
    with np.errstate(all="ignore"):
        s_ratio = np.clip(np.exp(lamt - s_mean), 0, 1e6)
        r_ratio = np.clip(np.exp(lamt - r_mean), 0, 1e6)
        s_z = np.where(np.isfinite(s_std), (lamt - s_mean) / (s_std + 1e-3), np.nan)
        s_fan = np.where(sh_c > 0, s_out[:, 0] / sh_c, np.nan)
        r_fan = np.where(rh_c > 0, r_in[:, 0] / rh_c, np.nan)
        pair_share = np.where(sh_c > 0, pair_c / sh_c, np.nan)
        passthru = np.clip(amt / (sin24[:, 0] + 1.0), 0, 10)

    hour = (ts_abs // 3600) % 24
    dow = ((ts_abs // 86400) + 3) % 7                      # Monday=0 (1970-01-01 was a Thursday)
    tmap = type_map or {}
    tcode = (df["transaction_type"].astype(str).map(tmap).fillna(-1).to_numpy(np.float64)
             if "transaction_type" in df.columns else np.full(n, -1.0))
    ccode = (df["currency"].astype(str).map(currency_map or {}).fillna(-1).to_numpy(np.float64)
             if "currency" in df.columns else np.full(n, -1.0))
    feats = {
        "amount": amt, "log_amount": lamt, "hour": hour, "day_of_week": dow,
        "is_weekend": (dow >= 5), "is_night": (hour < 6), "type_code": tcode, "currency_code": ccode,
        "is_self_transfer": (s == r),
        "s_cnt_5m": c5, "s_cnt_1h": c1, "s_cnt_24h": c24, "s_amt_1h": a1[:, 0], "s_amt_24h": a24[:, 0],
        "s_hist_cnt": sh_c, "s_hist_mean_log": s_mean, "s_hist_std_log": s_std, "s_amt_ratio": s_ratio, "s_amt_z": s_z,
        "s_secs_since_last": E_s.last_gap(s, ts),
        "r_hist_cnt": rh_c, "r_hist_mean_log": r_mean, "r_amt_ratio": r_ratio, "r_in_cnt_1h": rin1, "r_in_cnt_24h": rin24,
        "s_out_deg": s_out[:, 0], "s_in_deg": s_in[:, 0], "r_in_deg": r_in[:, 0], "r_out_deg": r_out[:, 0],
        "s_fanout_ratio": s_fan, "r_fanin_ratio": r_fan,
        "is_new_pair": (pair_c == 0), "pair_hist_cnt": pair_c, "pair_share": pair_share, "reverse_pair_cnt": rev_c,
        "s_inflow_1h": sin1[:, 0], "s_inflow_24h": sin24[:, 0], "pass_through_24h": passthru,
        "rapid_move_flag": ((sin1[:, 0] > 0) & (amt >= 0.8 * sin1[:, 0])),
    }
    out = pd.DataFrame({c: np.asarray(feats[c], dtype=np.float32) for c in FEATURE_COLUMNS}, index=df.index)
    return out.replace([np.inf, -np.inf], np.nan)


# ---- account-level features (aggregations of history up to as_of) --------------------
ACCOUNT_FEATURES = ["in_cnt", "out_cnt", "in_amt", "out_amt", "net_flow", "uniq_senders", "uniq_receivers",
                    "avg_amt", "median_amt", "std_log_amt", "txn_per_day", "max_share_out", "max_share_in",
                    "pass_through", "rapid_out_frac", "fan_in_ratio", "fan_out_ratio"]


def build_account_features(df: pd.DataFrame, as_of: Optional[int] = None) -> pd.DataFrame:
    if as_of is not None:
        df = df[df["ts"] <= as_of]
    if len(df) == 0:
        return pd.DataFrame(columns=ACCOUNT_FEATURES, dtype=float)
    s, r, cats = _encode_ids(df["sender_id"], df["receiver_id"])
    A = len(cats)
    amt = df["amount"].to_numpy(np.float64); ts = df["ts"].to_numpy(np.int64); ts0 = ts - ts.min()
    lamt = np.log1p(amt)
    bc = lambda idx, w=None: np.bincount(idx, weights=w, minlength=A).astype(np.float64)
    out_c, in_c, out_a, in_a = bc(s), bc(r), bc(s, amt), bc(r, amt)
    pc = s * A + r
    upc, pcnt = np.unique(pc, return_counts=True)
    ps, pr = upc // A, upc % A
    uniq_recv, uniq_send = bc(ps), bc(pr)
    max_out = pd.Series(pcnt).groupby(ps).max().reindex(range(A)).to_numpy(dtype=float)
    max_in = pd.Series(pcnt).groupby(pr).max().reindex(range(A)).to_numpy(dtype=float)
    both = np.concatenate([s, r])
    long = pd.DataFrame({"a": both, "t": np.concatenate([ts, ts]), "x": np.concatenate([amt, amt])})
    g = long.groupby("a")
    med = g["x"].median().reindex(range(A)).to_numpy(float)
    tmin, tmax = g["t"].min().reindex(range(A)).to_numpy(float), g["t"].max().reindex(range(A)).to_numpy(float)
    sl = bc(s, lamt) + bc(r, lamt); sl2 = bc(s, lamt ** 2) + bc(r, lamt ** 2)
    tot = out_c + in_c
    big = int(ts0.max()) + 8 * 86400 + 10
    E_r = _Events(r, ts0, [amt], big)
    rin_cnt, _ = E_r.query(s, ts0, 3600)
    rapid = bc(s, (rin_cnt > 0).astype(float))
    with np.errstate(all="ignore"):
        mean_l = sl / tot
        std_l = np.sqrt(np.maximum(sl2 / tot - mean_l ** 2, 0))
        days = np.maximum((tmax - tmin) / 86400.0, 1.0)
        feat = pd.DataFrame({
            "in_cnt": in_c, "out_cnt": out_c, "in_amt": in_a, "out_amt": out_a, "net_flow": in_a - out_a,
            "uniq_senders": uniq_send, "uniq_receivers": uniq_recv, "avg_amt": (in_a + out_a) / tot,
            "median_amt": med, "std_log_amt": std_l, "txn_per_day": tot / days,
            "max_share_out": max_out / out_c, "max_share_in": max_in / in_c,
            "pass_through": np.minimum(in_a, out_a) / np.maximum(np.maximum(in_a, out_a), 1e-9),
            "rapid_out_frac": rapid / out_c, "fan_in_ratio": uniq_send / in_c, "fan_out_ratio": uniq_recv / out_c,
        }, index=pd.Index(cats.astype(str), name="account_id"))
    feat = feat[tot > 0]                      # drop categories with no activity in this slice
    return feat.fillna({"max_share_out": 0, "max_share_in": 0, "rapid_out_frac": 0, "fan_in_ratio": 0, "fan_out_ratio": 0})


# ---- liquidity features ------------------------------------------------------------
LIQ_FEATURES = ["in_1h", "out_1h", "net_1h", "in_24h", "out_24h", "net_24h", "in_7d", "out_7d", "net_7d",
                "io_ratio_24h", "outflow_spike", "net_vol_daily", "net_z_24h", "funding_hhi_7d",
                "withdrawal_pressure", "sudden_outflow", "cum_net_proxy", "baseline_days"]


def compute_liquidity_features(df: pd.DataFrame, as_of: Optional[int] = None) -> pd.DataFrame:
    """Entity (account) level cash-flow features as of `as_of`. Uses ONLY rows with ts <= as_of.
    NOTE: no balance column is used (post-txn balances leak); balance is proxied by cumulative net flow."""
    if len(df) == 0:
        return pd.DataFrame(columns=LIQ_FEATURES, dtype=float)
    as_of = int(df["ts"].max()) if as_of is None else int(as_of)
    df = df[df["ts"] <= as_of]
    s, r, cats = _encode_ids(df["sender_id"], df["receiver_id"])
    A = len(cats)
    amt = df["amount"].to_numpy(np.float64); ts = df["ts"].to_numpy(np.int64)
    e = np.concatenate([s, r]); t = np.concatenate([ts, ts]); flow = np.concatenate([-amt, amt])
    cp = np.concatenate([r, s]); age = as_of - t
    inflow = flow > 0
    bc = lambda m, w: np.bincount(e[m], weights=w[m], minlength=A).astype(np.float64)
    res = {}
    for name, sec in (("1h", 3600), ("24h", 86400), ("7d", 7 * 86400)):
        w = age < sec
        res[f"in_{name}"] = bc(w & inflow, flow)
        res[f"out_{name}"] = bc(w & ~inflow, -flow)
        res[f"net_{name}"] = res[f"in_{name}"] - res[f"out_{name}"]
    # baseline: the 30 days BEFORE the last 24h (days_ago in [1, 30])
    days_ago = age // 86400
    mb = (days_ago >= 1) & (days_ago <= 30)
    first_ts = pd.Series(t).groupby(e).min().reindex(range(A)).to_numpy(float)
    base_days = np.clip(np.floor((as_of - 86400 - first_ts) / 86400) + 1, 0, 30)
    base_days = np.where(np.isnan(base_days), 0, base_days)
    day_df = pd.DataFrame({"e": e[mb], "d": days_ago[mb], "net": flow[mb], "out": np.where(inflow[mb], 0.0, -flow[mb])})
    day_g = day_df.groupby(["e", "d"]).sum().reset_index()
    sum_out = np.bincount(day_g.e.to_numpy(), weights=day_g.out.to_numpy(), minlength=A)
    sum_net = np.bincount(day_g.e.to_numpy(), weights=day_g.net.to_numpy(), minlength=A)
    sum_net2 = np.bincount(day_g.e.to_numpy(), weights=day_g.net.to_numpy() ** 2, minlength=A)
    # funding concentration (HHI of 7d inflow by counterparty)
    w7 = (age < 7 * 86400) & inflow
    cpf = pd.DataFrame({"e": e[w7], "cp": cp[w7], "x": flow[w7]}).groupby(["e", "cp"])["x"].sum().reset_index()
    tot_in = np.bincount(cpf.e.to_numpy(), weights=cpf.x.to_numpy(), minlength=A)
    hhi = np.bincount(cpf.e.to_numpy(), weights=(cpf.x.to_numpy() / np.maximum(tot_in[cpf.e.to_numpy()], 1e-9)) ** 2, minlength=A)
    cum_net = np.bincount(e, weights=flow, minlength=A)
    cum_gross = np.bincount(e, weights=np.abs(flow), minlength=A)
    with np.errstate(all="ignore"):
        bd = np.where(base_days > 0, base_days, np.nan)
        mean_out = sum_out / bd
        mean_net = sum_net / bd
        vol = np.sqrt(np.maximum(sum_net2 / bd - mean_net ** 2, 0))
        out = pd.DataFrame(res)
        out["io_ratio_24h"] = out.in_24h / (out.out_24h + 1e-9)
        out["outflow_spike"] = np.where(base_days >= 3, out.out_24h / (mean_out + 1e-9), np.nan)
        out["net_vol_daily"] = np.where(base_days >= 3, vol, np.nan)
        out["net_z_24h"] = np.where(base_days >= 3, (out.net_24h - mean_net) / (vol + 1e-9), np.nan)
        out["funding_hhi_7d"] = hhi
        out["withdrawal_pressure"] = out.out_24h / (out.in_24h + out.out_24h + 1e-9)
        out["sudden_outflow"] = ((out.out_1h > 0.5 * out.out_24h) & (out.out_1h > 0) &
                                 (out.out_24h > 3 * (out.out_7d / 7.0))).astype(float)
        out["cum_net_proxy"] = cum_net / (cum_gross + 1e-9)
        out["baseline_days"] = base_days
    out.index = pd.Index(cats.astype(str), name="entity_id")
    out = out[cum_gross > 0]                  # only entities with activity up to as_of
    return out.replace([np.inf, -np.inf], np.nan)[LIQ_FEATURES]


LIQ_WEIGHTS = {"outflow_dominance": 0.30, "outflow_spike": 0.25, "net_outflow_shock": 0.20,
               "funding_concentration": 0.10, "sudden_burst": 0.15}
LIQ_THRESHOLDS = {"medium": 0.35, "high": 0.55, "critical": 0.75}   # policy-set, NOT learned


def liquidity_stress_score(f: pd.DataFrame) -> pd.DataFrame:
    """DETERMINISTIC stress score in [0,1]. Not a trained model; weights are documented policy choices."""
    c = pd.DataFrame(index=f.index)
    active = (f.out_24h > 0)
    c["outflow_dominance"] = np.where(active, np.clip((f.out_24h / (f.in_24h + 1e-9) - 1.0) / 2.0, 0, 1), 0.0)
    c["outflow_spike"] = np.where(f.outflow_spike.notna(), np.clip((f.outflow_spike - 1.0) / 4.0, 0, 1), 0.0)
    c["net_outflow_shock"] = np.where(f.net_z_24h.notna(), np.clip(-f.net_z_24h / 3.0, 0, 1), 0.0)
    c["funding_concentration"] = np.where(f.in_7d > 0, np.clip((f.funding_hhi_7d - 0.25) / 0.75, 0, 1), 0.0)
    burst = f.out_1h / (f.out_24h + 1e-9)
    c["sudden_burst"] = np.where(active, np.clip((burst - 1 / 24) / 0.5, 0, 1), 0.0)
    score = sum(LIQ_WEIGHTS[k] * c[k] for k in LIQ_WEIGHTS)
    c["stress_score"] = np.clip(score, 0, 1)
    return c


# ======================================================================================
# 5. MODEL TRAINING
# ======================================================================================


def _make_gbm(pos_weight: float, cfg: Config):
    if HAVE_LGBM:
        return lgb.LGBMClassifier(n_estimators=cfg.n_estimators, learning_rate=0.05, num_leaves=63,
                                  min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                  reg_lambda=1.0, scale_pos_weight=pos_weight, n_jobs=-1, random_state=cfg.seed,
                                  verbosity=-1), "lightgbm"
    return HistGradientBoostingClassifier(max_iter=min(cfg.n_estimators, 300), learning_rate=0.08,
                                          class_weight="balanced", random_state=cfg.seed), "sklearn_hist_gbm"


def _make_lr(cfg: Config):
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("slog", FunctionTransformer(_slog1p)),
                     ("sc", StandardScaler()),
                     ("lr", LogisticRegression(max_iter=300, class_weight="balanced", C=0.5))])


def _importance(model, cols) -> pd.DataFrame:
    if hasattr(model, "feature_importances_"):
        imp = model.feature_importances_.astype(float)
    elif hasattr(model, "named_steps"):
        imp = np.abs(model.named_steps["lr"].coef_[0])
    else:
        imp = np.zeros(len(cols))
    return pd.DataFrame({"feature": cols, "importance": imp}).sort_values("importance", ascending=False)


def _proba(model, X):
    return model.predict_proba(X)[:, 1]


def train_binary(Xtr, ytr, Xva, yva, Xte, yte, cols: List[str], cfg: Config, name: str, Xtr_eval=None, ytr_eval=None):
    """LR baseline + GBM; pick by VALIDATION PR-AUC; isotonic calibration + thresholds on VALIDATION;
    TEST metrics computed once at the end."""
    npos = int(ytr.sum()); nneg = int(len(ytr) - npos)
    pw = float(min(max(nneg / max(npos, 1), 1.0), 50.0))
    models = {}
    lr = _make_lr(cfg).fit(Xtr[cols], ytr); models["logistic_regression"] = lr
    gbm, kind = _make_gbm(pw, cfg)
    if kind == "lightgbm" and yva.sum() > 0:
        gbm.fit(Xtr[cols], ytr, eval_set=[(Xva[cols], yva)], eval_metric="average_precision",
                callbacks=[lgb.early_stopping(50, verbose=False)])
    else:
        gbm.fit(Xtr[cols], ytr)
    models[kind] = gbm
    val_pr = {k: (average_precision_score(yva, _proba(m, Xva[cols])) if 0 < yva.sum() < len(yva) else -1)
              for k, m in models.items()}
    best = max(val_pr, key=val_pr.get)
    model = models[best]
    pva_raw = _proba(model, Xva[cols])
    cal = None
    if yva.sum() >= 20:
        cal = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(pva_raw, yva)
    f = (lambda p: cal.predict(p)) if cal else (lambda p: p)
    pva = f(pva_raw)
    th = choose_thresholds(yva, pva)
    res = {"best_model": best, "val_pr_auc_by_model": val_pr, "thresholds": th, "calibrated": cal is not None,
           "pos_weight": pw, "models": {}}
    curves = {}
    for k, m in models.items():
        pte_raw = _proba(m, Xte[cols])
        pv = _proba(m, Xva[cols])
        if k == best:
            pte = f(pte_raw)
            res["models"][k] = {"val": eval_scores(yva, pva, cfg.top_k, th["high"]),
                                "test": eval_scores(yte, pte, cfg.top_k, th["high"]), "selected": True}
            if yte.sum() > 0:
                try:
                    fr, mp = calibration_curve(yte, pte, n_bins=10, strategy="quantile")
                    res["calibration_test"] = {"frac_pos": fr.tolist(), "mean_pred": mp.tolist()}
                except Exception:
                    pass
        else:
            res["models"][k] = {"val": eval_scores(yva, pv, cfg.top_k), "test": eval_scores(yte, pte_raw, cfg.top_k),
                                "selected": False}
        if Xtr_eval is not None:      # TRAIN metrics on a NATURAL-balance sample of the training period
            ptr = _proba(m, Xtr_eval[cols])
            res["models"][k]["train"] = (eval_scores(ytr_eval, f(ptr), cfg.top_k, th["high"]) if k == best
                                         else eval_scores(ytr_eval, ptr, cfg.top_k))
        curves[k] = pte_raw if k != best else pte
    res["importance"] = _importance(model, cols)
    return model, cal, res, curves


# ---- persistence / registry ------------------------------------------------------------


def _new_version() -> str:
    return datetime.now(timezone.utc).strftime("v%Y%m%d%H%M%S")


def save_bundle(cfg: Config, name: str, bundle: Dict[str, Any], metrics: Dict[str, Any], extra_meta: Dict[str, Any],
                dataset_version: str) -> str:
    version = bundle["model_version"]
    d = Path(cfg.out_dir) / "models" / name / version
    d.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, d / "pipeline.joblib", compress=3)
    meta = {"model_name": name, "model_version": version, "schema_version": SCHEMA_VERSION,
            "training_date": datetime.now(timezone.utc).isoformat(), "dataset_version": dataset_version,
            "features": bundle.get("feature_order", []), "metrics": metrics,
            "thresholds": bundle.get("thresholds", {}), **extra_meta}
    (d / "metadata.json").write_text(json.dumps(_clean(meta), indent=2))
    (d / "feature_order.json").write_text(json.dumps(bundle.get("feature_order", [])))
    (Path(cfg.out_dir) / "models" / name / "LATEST").write_text(version)
    reg_p = Path(cfg.out_dir) / "model_registry.json"
    reg = json.loads(reg_p.read_text()) if reg_p.exists() else {"models": []}
    reg["models"] = [m for m in reg["models"] if not (m["model_name"] == name and m["model_version"] == version)]
    reg["models"].append({k: meta[k] for k in ("model_name", "model_version", "training_date", "features", "metrics",
                                                "dataset_version", "thresholds")} |
                         {"path": str(d / "pipeline.joblib"), "scoring_method": extra_meta.get("scoring_method")})
    reg_p.write_text(json.dumps(_clean(reg), indent=2))
    return version


class AccountScorer:
    """Account RISK SCORING engine (heuristic + IsolationForest). NOT a supervised classifier."""
    SIGNALS = {"fan_in": "uniq_senders", "fan_out": "uniq_receivers", "volume": "total_amt",
               "volatility": "std_log_amt"}
    W = {"fan_in": 0.20, "fan_out": 0.20, "pass_through": 0.25, "rapid_out": 0.20, "volume": 0.10, "volatility": 0.05}

    def fit(self, feat: pd.DataFrame, seed=42, max_iso=300_000):
        f = self._prep(feat)
        grid = np.linspace(0, 1, 101)
        self.grid = grid
        self.ref = {c: np.quantile(f[c].to_numpy(), grid) for c in ["uniq_senders", "uniq_receivers", "total_amt", "std_log_amt"]}
        X = f[ACCOUNT_FEATURES]
        samp = X.sample(min(len(X), max_iso), random_state=seed)
        self.iso = Pipeline([("imp", SimpleImputer(strategy="median")), ("slog", FunctionTransformer(_slog1p)),
                             ("if", IsolationForest(n_estimators=150, max_samples=256, random_state=seed, n_jobs=-1))]).fit(samp)
        raw = -self.iso.decision_function(X)
        self.iso_grid = np.quantile(raw, grid)
        sc = self.score(feat)["risk_score"].to_numpy()
        self.thresholds = {"medium": float(np.quantile(sc, 0.90)), "high": float(np.quantile(sc, 0.98)),
                           "critical": float(np.quantile(sc, 0.998))}
        return self

    @staticmethod
    def _prep(feat):
        f = feat.copy()
        f["total_amt"] = f["in_amt"] + f["out_amt"]
        return f

    def score(self, feat: pd.DataFrame) -> pd.DataFrame:
        f = self._prep(feat)
        pct = lambda c: np.interp(f[c].to_numpy(float), self.ref[c], self.grid)
        gate = ((f.in_cnt >= 3) & (f.out_cnt >= 3)).to_numpy(float)
        comp = pd.DataFrame({
            "fan_in": pct("uniq_senders"), "fan_out": pct("uniq_receivers"),
            "pass_through": f["pass_through"].to_numpy(float) * gate,
            "rapid_out": f["rapid_out_frac"].to_numpy(float),
            "volume": pct("total_amt"), "volatility": pct("std_log_amt")}, index=f.index)
        det = sum(self.W[k] * comp[k] for k in self.W)
        iso = np.interp(-self.iso.decision_function(f[ACCOUNT_FEATURES]), self.iso_grid, self.grid)
        out = comp.copy()
        out["deterministic_score"] = det
        out["isolation_score"] = iso
        out["risk_score"] = np.clip(0.6 * det + 0.4 * iso, 0, 1)
        return out


def _signal_refs(X: pd.DataFrame) -> Dict[str, float]:
    return {c: float(np.nanquantile(X[c].to_numpy(), 0.99)) for c in SIGNAL_TEXT if c in X}


# ---- synthetic credit dataset (ONLY used when no real credit data exists) --------------
CREDIT_FEATURES = ["loan_amount", "outstanding_amount", "annual_income", "monthly_debt_payment", "credit_utilization",
                   "delinquencies_12m", "on_time_payment_ratio", "tenure_months", "debt_to_income",
                   "outstanding_ratio"]


def _credit_engineer(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    d["debt_to_income"] = (d["monthly_debt_payment"] * 12) / d["annual_income"].clip(lower=1)
    d["outstanding_ratio"] = d["outstanding_amount"] / d["loan_amount"].clip(lower=1)
    return d


def make_synthetic_credit(n: int, seed: int = 42) -> pd.DataFrame:
    """SYNTHETIC. Default is generated from the features via an ASSUMED logistic function, so metrics only
    validate the pipeline, NOT real-world predictive power. Do not use for lending decisions."""
    rng = np.random.default_rng(seed)
    income = rng.lognormal(12.6, 0.5, n)
    loan = income * rng.uniform(0.3, 3.0, n)
    out = loan * rng.uniform(0.1, 1.0, n)
    debt = income / 12 * rng.beta(2, 6, n)
    util = np.clip(rng.beta(2, 3, n), 0, 1)
    delq = rng.poisson(0.4, n)
    ontime = np.clip(rng.beta(8, 1.5, n), 0, 1)
    ten = rng.integers(3, 120, n)
    d = pd.DataFrame(dict(loan_amount=loan, outstanding_amount=out, annual_income=income, monthly_debt_payment=debt,
                          credit_utilization=util, delinquencies_12m=delq, on_time_payment_ratio=ontime,
                          tenure_months=ten))
    d = _credit_engineer(d)
    z = (-3.2 + 2.0 * d.credit_utilization + 2.5 * d.debt_to_income.clip(upper=2) + 0.45 * d.delinquencies_12m
         - 2.8 * d.on_time_payment_ratio + 0.6 * d.outstanding_ratio - 0.004 * d.tenure_months + 3.0)
    d["default"] = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    return d


# ======================================================================================
# 6. MAIN TRAINING PIPELINE
# ======================================================================================


def _save_plots(cfg: Config, name: str, yte, curves: Dict[str, np.ndarray], res: Dict, imp: pd.DataFrame):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        rep = Path(cfg.out_dir) / "reports"; rep.mkdir(parents=True, exist_ok=True)
        if yte.sum() > 0:
            plt.figure(figsize=(6, 4.5))
            for k, p in curves.items():
                pr, rc, _ = precision_recall_curve(yte, p)
                plt.plot(rc, pr, label=f"{k} (AP={average_precision_score(yte, p):.3f})")
            plt.axhline(yte.mean(), ls="--", c="grey", label="base rate")
            plt.xlabel("recall"); plt.ylabel("precision"); plt.title(f"{name}: PR curve (TEST)"); plt.legend()
            plt.tight_layout(); plt.savefig(rep / f"{name}_pr_curve.png", dpi=110); plt.close()
        top = imp.head(15).iloc[::-1]
        plt.figure(figsize=(6, 5)); plt.barh(top.feature, top.importance)
        plt.title(f"{name}: top feature importance"); plt.tight_layout()
        plt.savefig(rep / f"{name}_feature_importance.png", dpi=110); plt.close()
        imp.to_csv(rep / f"{name}_feature_importance.csv", index=False)
    except Exception as e:  # plotting must never kill the run
        print("[plot warning]", e)


def _metrics_table(res: Dict) -> pd.DataFrame:
    rows = []
    for k, v in res["models"].items():
        for split in ("train", "val", "test"):
            if split not in v:
                continue
            m = v[split]
            rows.append({"model": k + (" *" if v["selected"] else ""), "split": split,
                         **{x: m.get(x) for x in ["pr_auc", "roc_auc", "precision", "recall", "f1", "accuracy"]},
                         **{f"P@{k_}": m.get(f"precision@{k_}") for k_ in (50, 100, 500)},
                         **{f"R@{k_}": m.get(f"recall@{k_}") for k_ in (50, 100, 500)}})
    return pd.DataFrame(rows).round(4)


def run_pipeline(data_path: str, cfg: Optional[Config] = None) -> Dict[str, Any]:
    cfg = cfg or Config()
    t0 = time.time()
    Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
    rep_dir = Path(cfg.out_dir) / "reports"; rep_dir.mkdir(exist_ok=True)

    # ---------- 1. load + inspect ----------
    df, report = load_dataset(data_path, cfg)
    dv = report["dataset_version"]
    has_label = report["label_available"]

    # ---------- 2. time-based split ----------
    t1 = float(df.ts.quantile(cfg.train_frac)); t2 = float(df.ts.quantile(cfg.train_frac + cfg.val_frac))
    itr = (df.ts < t1).to_numpy(); iva = ((df.ts >= t1) & (df.ts < t2)).to_numpy(); ite = (df.ts >= t2).to_numpy()
    periods = {"train": [_ts_str(df.ts[itr].min()), _ts_str(df.ts[itr].max()), int(itr.sum())],
               "validation": [_ts_str(df.ts[iva].min()), _ts_str(df.ts[iva].max()), int(iva.sum())],
               "test": [_ts_str(df.ts[ite].min()), _ts_str(df.ts[ite].max()), int(ite.sum())]}
    _show("TIME-BASED SPLIT [start, end, rows]", pd.DataFrame(periods, index=["start", "end", "rows"]).T)
    if has_label:
        for nm, m in (("train", itr), ("validation", iva), ("test", ite)):
            print(f"  positives in {nm}: {int(df.label[m].sum()):,} / {int(m.sum()):,}")
    report["periods"] = periods

    # ---------- 3. features (past-only) ----------
    type_map = build_type_map(df.transaction_type[itr])
    currency_map = build_type_map(df.currency[itr])
    t_f = time.time()
    X = build_transaction_features(df, type_map, currency_map)
    print(f"features built: {X.shape} in {time.time()-t_f:.0f}s | memory {X.memory_usage().sum()/1e6:.0f} MB")
    _show("FEATURE LIST", pd.Series(FEATURE_COLUMNS, name="feature"))
    summary: Dict[str, Any] = {"periods": periods, "dataset_version": dv, "models_trained": [], "models_skipped": {}}

    # ---------- 4. MODEL 1: supervised fraud / AML ----------
    sem = "laundering" if (report["label_column"] and "launder" in _norm(report["label_column"])) else "fraud"
    risk_type = "TRANSACTION_AML" if sem == "laundering" else "TRANSACTION_FRAUD"
    if has_label and df.label[itr].sum() >= 10 and df.label[ite].sum() >= 1:
        y = df.label.to_numpy()
        tr_idx = np.where(itr)[0]
        if len(tr_idx) > cfg.max_train_rows:                     # L10: down-sample negatives, TRAIN only
            rng = np.random.default_rng(cfg.seed)
            pos = tr_idx[y[tr_idx] == 1]; neg = tr_idx[y[tr_idx] == 0]
            keep_neg = rng.choice(neg, size=max(cfg.max_train_rows - len(pos), 1), replace=False)
            tr_idx = np.sort(np.concatenate([pos, keep_neg]))
            print(f"[memory] train down-sampled to {len(tr_idx):,} rows (all {len(pos):,} positives kept).")
        va_idx = np.where(iva)[0]
        if len(va_idx) > cfg.max_val_rows:
            pos = va_idx[y[va_idx] == 1]; neg = va_idx[y[va_idx] == 0]
            va_idx = np.sort(np.concatenate([pos, np.random.default_rng(1).choice(neg, cfg.max_val_rows - len(pos), replace=False)])) \
                if len(pos) < cfg.max_val_rows else va_idx
        te_idx = np.where(ite)[0]
        nat = np.where(itr)[0]
        nat = np.sort(np.random.default_rng(7).choice(nat, min(len(nat), 500_000), replace=False))
        model, cal, res, curves = train_binary(X.iloc[tr_idx], y[tr_idx], X.iloc[va_idx], y[va_idx],
                                               X.iloc[te_idx], y[te_idx], FEATURE_COLUMNS, cfg, "fraud_model",
                                               Xtr_eval=X.iloc[nat], ytr_eval=y[nat])
        _show(f"MODEL 1 ({risk_type}) METRICS  (* = selected)", _metrics_table(res))
        _show("TEST CONFUSION MATRIX @ HIGH threshold", pd.Series(res["models"][res["best_model"]]["test"]["confusion_matrix"]))
        _show("TOP FEATURES", res["importance"].head(15))
        _save_plots(cfg, "fraud_model", y[te_idx], curves, res, res["importance"])
        if res["models"][res["best_model"]]["test"].get("recall", 0) == 0:
            print("[WARN] model catches 0 test positives at the HIGH threshold -> likely very few/unlearnable positives.")
        ver = _new_version()
        bundle = {"model": model, "calibrator": cal, "feature_order": FEATURE_COLUMNS, "type_map": type_map,
                  "thresholds": res["thresholds"], "signal_refs": _signal_refs(X.iloc[tr_idx]), "model_version": ver, "currency_map": currency_map,
                  "risk_type": risk_type, "label_semantics": sem, "model_kind": res["best_model"]}
        mt = {k: v for k, v in res.items() if k not in ("importance",)}
        save_bundle(cfg, "fraud_model", bundle, mt,
                    {"scoring_method": "supervised_classifier", "label_column": report["label_column"],
                     "label_semantics": sem, "training_period": periods["train"][:2],
                     "validation_period": periods["validation"][:2], "test_period": periods["test"][:2],
                     "description": f"Supervised classifier of the dataset's '{report['label_column']}' label at transaction level."},
                    dv)
        sel = res["models"][res["best_model"]]
        summary["fraud_model"] = {"version": ver, "best": res["best_model"], "test": sel["test"],
                                  "metrics": {"train": sel.get("train"), "val": sel["val"], "test": sel["test"]},
                                  "baseline_test": res["models"].get("logistic_regression", {}).get("test"),
                                  "label_semantics": sem, "label_column": report["label_column"]}
        summary["models_trained"].append("fraud_model")
        del model, curves; gc.collect()
    else:
        why = "no usable binary label column" if not has_label else "too few positives in train/test period"
        summary["models_skipped"]["fraud_model"] = why
        print(f"[SKIP] Model 1 (supervised): {why}. No target was fabricated.")

    # ---------- 5. Anomaly detector (unsupervised) ----------
    tr_i = np.where(itr)[0]
    samp_i = np.random.default_rng(cfg.seed).choice(tr_i, size=min(len(tr_i), 200_000), replace=False)
    iso = Pipeline([("imp", SimpleImputer(strategy="median")), ("slog", FunctionTransformer(_slog1p)),
                    ("if", IsolationForest(n_estimators=150, max_samples=256, random_state=cfg.seed, n_jobs=-1))])
    iso.fit(X.iloc[samp_i][ANOMALY_FEATURES])
    raw_tr = -iso.decision_function(X.iloc[samp_i][ANOMALY_FEATURES])
    grid = np.linspace(0, 1, 201)
    an_grid = np.quantile(raw_tr, grid)
    an_th = {"medium": 0.95, "high": 0.99, "critical": 0.999}     # percentile-of-train anomaly score
    metrics_an: Dict[str, Any] = {"note": "unsupervised; score is a percentile vs train distribution, NOT a probability"}
    if has_label and df.label[ite].sum() > 0:
        te_i = np.where(ite)[0]
        if len(te_i) > 500_000:
            te_i = np.sort(np.random.default_rng(3).choice(te_i, 500_000, replace=False))
        sc = np.interp(-iso.decision_function(X.iloc[te_i][ANOMALY_FEATURES]), an_grid, grid)
        metrics_an["diagnostic_vs_label_test"] = eval_scores(df.label.to_numpy()[te_i], sc, cfg.top_k)
        print("anomaly diagnostic (NOT fraud probability) test PR-AUC:", metrics_an["diagnostic_vs_label_test"]["pr_auc"])
    ver = _new_version()
    save_bundle(cfg, "anomaly_model",
                {"model": iso, "score_grid": an_grid, "grid": grid, "feature_order": ANOMALY_FEATURES,
                 "type_map": type_map, "currency_map": currency_map, "thresholds": an_th, "model_version": ver},
                metrics_an, {"scoring_method": "unsupervised_anomaly", "training_period": periods["train"][:2],
                             "description": "IsolationForest on behavioural features. Output is anomaly_score, never fraud probability."}, dv)
    summary["models_trained"].append("anomaly_model")
    summary["anomaly"] = metrics_an
    del iso; gc.collect()

    # ---------- 6. Account AML risk SCORING (+ honest back-test) ----------
    cutoff = int(t2 - 1)
    af = build_account_features(df, as_of=cutoff)
    scorer = AccountScorer().fit(af, seed=cfg.seed)
    acc_metrics: Dict[str, Any] = {"n_accounts_scored": int(len(af)), "method": "heuristic risk scoring + IsolationForest"}
    if has_label and df.label[ite].sum() > 0:
        tst = df[ite & (df.label.to_numpy() == 1)]
        bad = set(tst.sender_id.astype(str)) | set(tst.receiver_id.astype(str))
        sc_all = scorer.score(af)["risk_score"]
        ya = af.index.isin(list(bad)).astype(int)
        acc_metrics["backtest_vs_future_labels"] = eval_scores(ya, sc_all.to_numpy(), cfg.top_k)
        acc_metrics["backtest_note"] = ("Scores use history <= end of validation; 'positive' = account touched a labelled "
                                        "transaction in the TEST period. Validation of a heuristic, not training.")
        print("account-risk back-test PR-AUC:", acc_metrics["backtest_vs_future_labels"]["pr_auc"],
              "| base rate", acc_metrics["backtest_vs_future_labels"]["base_rate"])
    save_bundle(cfg, "account_risk_model",
                {"scorer": scorer, "feature_order": ACCOUNT_FEATURES, "thresholds": scorer.thresholds,
                 "model_version": _new_version()}, acc_metrics,
                {"scoring_method": "risk_scoring_heuristic", "description":
                 "RISK SCORING (not supervised prediction): no account-level labels exist. Risk levels are rank-based "
                 "(p90/p98/p99.8 of the reference population)."}, dv)
    summary["models_trained"].append("account_risk_model")
    summary["account_risk"] = acc_metrics
    top_acc = scorer.score(af).sort_values("risk_score", ascending=False).head(10)
    _show("TOP-10 RISKIEST ACCOUNTS (scoring, not labels)", top_acc[["risk_score", "deterministic_score", "isolation_score"]].round(3))
    del af; gc.collect()

    # ---------- 7. Liquidity (deterministic stress score) ----------
    lf = compute_liquidity_features(df)          # as_of = end of data
    lc = liquidity_stress_score(lf)
    pop = lc.stress_score.describe(percentiles=[.5, .9, .99]).round(4).to_dict()
    liq_metrics = {"approach": "deterministic stress score (no liquidity label exists; not a trained model)",
                   "n_entities": int(len(lf)), "stress_score_distribution": pop,
                   "level_counts": pd.cut(lc.stress_score, [-1, LIQ_THRESHOLDS["medium"], LIQ_THRESHOLDS["high"],
                                                            LIQ_THRESHOLDS["critical"], 2],
                                          labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"]).value_counts().to_dict()}
    save_bundle(cfg, "liquidity_model",
                {"weights": LIQ_WEIGHTS, "feature_order": LIQ_FEATURES, "thresholds": LIQ_THRESHOLDS,
                 "model_version": _new_version()}, liq_metrics,
                {"scoring_method": "deterministic_stress_score",
                 "description": "Rule-based liquidity stress score at account/entity level. Weights/thresholds are policy "
                                "settings to be calibrated by Treasury/ALM; not learned from data."}, dv)
    summary["models_trained"].append("liquidity_model")
    summary["liquidity"] = liq_metrics
    _show("LIQUIDITY STRESS (as of data end)", pd.Series(liq_metrics["level_counts"]))
    del lf, lc; gc.collect()

    # ---------- 8. Credit ----------
    cred_cols = report["credit_columns_detected"]
    print(f"[credit] columns in transaction file hinting at credit data: {cred_cols} "
          "(a transaction ledger has no borrower/loan/default fields -> not used)")
    try:
        _train_credit(cfg, dv, summary)
    except Exception as e:  # never let optional model kill the run
        summary["models_skipped"]["credit_model"] = f"failed: {e}"
        print("[credit warning]", e)

    # ---------- 9. reports ----------
    (rep_dir / "dataset_report.json").write_text(json.dumps(_clean(report), indent=2, default=str))
    (Path(cfg.out_dir) / "training_summary.json").write_text(json.dumps(_clean(summary), indent=2, default=str))
    write_readme(cfg, report, summary)
    configure(cfg.out_dir)
    print(build_scorecard(cfg, report, summary))
    print(f"\nDONE in {time.time()-t0:.0f}s. Artifacts in: {cfg.out_dir}")
    return summary


def _f(x, d=4):
    return "n/a" if x is None else (f"{x:.{d}f}" if isinstance(x, (int, float, np.floating)) else str(x))


def build_scorecard(cfg: Config, report: Dict, summary: Dict) -> str:
    """Plain-language scorecard: what was measured, what could NOT be measured, and why."""
    L = ["", "=" * 78, "RISK SENTINEL SCORECARD  (read this before trusting any number)", "=" * 78]
    fm = summary.get("fraud_model")
    L.append(f"\n[1] TRANSACTION FRAUD / AML  -- supervised, label = '{report.get('label_column')}'")
    if fm:
        L.append("    split        PR-AUC  ROC-AUC  precision  recall    F1   accuracy  P@100  R@500  lift-vs-random")
        for sp in ("train", "val", "test"):
            m = fm["metrics"].get(sp)
            if m:
                L.append(f"    {sp:<11s} {_f(m.get('pr_auc')):>7} {_f(m.get('roc_auc')):>8} {_f(m.get('precision')):>9} "
                         f"{_f(m.get('recall')):>7} {_f(m.get('f1')):>6} {_f(m.get('accuracy')):>9} "
                         f"{_f(m.get('precision@100'),3):>6} {_f(m.get('recall@500'),3):>6} {_f(m.get('lift_vs_random'),1):>8}x")
        t = fm["metrics"]["test"]
        L.append(f"    TEST base rate (share of true positives) = {_f(t['base_rate'], 5)}  -> a model that says "
                 f"'always legitimate' would score accuracy {_f(1 - t['base_rate'], 4)} yet catch NOTHING.")
        L.append("    => ACCURACY IS NOT THE RIGHT METRIC here. Judge by PR-AUC, recall, precision, P@K on TEST.")
        L.append("    TRAIN row is on the training period (natural balance); a big train>>test gap means over-fitting.")
        b = fm.get("baseline_test")
        if b:
            L.append(f"    Logistic-regression baseline TEST PR-AUC = {_f(b.get('pr_auc'))} (the selected model should beat it).")
        cm = t.get("confusion_matrix")
        if cm:
            L.append(f"    TEST confusion matrix @ HIGH threshold: TP={cm['tp']:,} FP={cm['fp']:,} FN={cm['fn']:,} TN={cm['tn']:,}")
    else:
        L.append(f"    NOT TRAINED: {summary['models_skipped'].get('fraud_model')}")

    L.append("\n[2] ACCOUNT AML RISK  -- heuristic RISK SCORING (no account labels exist; not supervised)")
    bt = (summary.get("account_risk") or {}).get("backtest_vs_future_labels")
    if bt:
        L.append(f"    Back-test (score from past history vs accounts touching labelled txns in the later TEST period):")
        L.append(f"    PR-AUC={_f(bt['pr_auc'])} vs random={_f(bt['base_rate'])} (lift {_f(bt.get('lift_vs_random'),1)}x), "
                 f"P@100={_f(bt.get('precision@100'),3)}, P@500={_f(bt.get('precision@500'),3)}")
        L.append("    This validates ranking quality of a heuristic. It is NOT a trained accuracy.")
    else:
        L.append("    No labels -> cannot be measured. Scores are rank-based screening signals only.")

    liq = summary.get("liquidity") or {}
    L.append("\n[3] LIQUIDITY RISK  -- deterministic stress score")
    L.append("    NO ground truth exists in a transaction ledger -> accuracy CANNOT be computed. Not a trained model.")
    L.append(f"    Entities scored: {liq.get('n_entities')}; level counts: {liq.get('level_counts')}")

    cr = summary.get("credit_model") or {}
    L.append("\n[4] CREDIT RISK")
    L.append(f"    Data used: {cr.get('data_source', 'n/a')}")
    if "SYNTHETIC" in str(cr.get("data_source", "")):
        L.append("    The transaction dataset has no borrower/loan/default fields. This model is a pipeline DEMO on synthetic data;")
        L.append("    its metrics say NOTHING about real credit risk. Provide a real credit CSV (Config.credit_csv) to get valid metrics.")
    L.append("\n[+] ANOMALY DETECTOR -- unsupervised score (percentile), not a probability and not an accuracy claim.")
    d = (summary.get("anomaly") or {}).get("diagnostic_vs_label_test")
    if d:
        L.append(f"    Diagnostic only: TEST PR-AUC vs label = {_f(d['pr_auc'])} (random = {_f(d['base_rate'])}).")
    L.append("=" * 78)
    txt = "\n".join(L)
    (Path(cfg.out_dir) / "reports").mkdir(parents=True, exist_ok=True)
    (Path(cfg.out_dir) / "reports" / "scorecard.txt").write_text(txt)
    return txt


def _train_credit(cfg: Config, dv: str, summary: Dict[str, Any]):
    if cfg.credit_csv:
        d = pd.read_csv(cfg.credit_csv)
        lab = next((c for c in d.columns if _norm(c) in ("default", "isdefault", "target", "label", "defaulted")), None)
        if lab is None:
            raise ValueError("credit_csv needs a binary default column named default/target/label")
        feats = [c for c in d.select_dtypes("number").columns if c != lab]
        src, method = "REAL credit data (stratified random split: add a time column + time split for production)", "supervised_classifier"
        d = d.rename(columns={lab: "default"})
    else:
        d = make_synthetic_credit(cfg.credit_n_synthetic, cfg.seed)
        feats = CREDIT_FEATURES
        src, method = ("SYNTHETIC data with ASSUMED default mechanism -> pipeline demonstration ONLY, "
                       "metrics do NOT reflect real-world performance"), "synthetic_demo_model"
    from sklearn.model_selection import train_test_split
    tr, tmp = train_test_split(d, test_size=0.4, stratify=d["default"], random_state=cfg.seed)
    va, te = train_test_split(tmp, test_size=0.5, stratify=tmp["default"], random_state=cfg.seed)
    model, cal, res, curves = train_binary(tr[feats], tr["default"].to_numpy(), va[feats], va["default"].to_numpy(),
                                           te[feats], te["default"].to_numpy(), feats, cfg, "credit_model")
    _show(f"MODEL 4 CREDIT METRICS ({src[:9]})", _metrics_table(res))
    _save_plots(cfg, "credit_model", te["default"].to_numpy(), curves, res, res["importance"])
    ver = _new_version()
    save_bundle(cfg, "credit_model",
                {"model": model, "calibrator": cal, "feature_order": feats, "thresholds": res["thresholds"],
                 "model_version": ver, "synthetic": method == "synthetic_demo_model"},
                {k: v for k, v in res.items() if k != "importance"},
                {"scoring_method": method, "data_source": src, "description": src}, dv + ("-synthetic" if method != "supervised_classifier" else ""))
    summary["models_trained"].append("credit_model")
    summary["credit_model"] = {"version": ver, "data_source": src, "test": res["models"][res["best_model"]]["test"]}


def write_readme(cfg: Config, report: Dict, summary: Dict):
    fm = summary.get("fraud_model")
    t = fm["test"] if fm else None
    txt = f"""# Risk Sentinel ML Layer — what each model predicts

Dataset: `{report['file']}` — {report['rows']:,} rows, {report['unique_accounts']:,} accounts,
{report['timestamp_min']} → {report['timestamp_max']}. Dataset version `{report['dataset_version']}`.
Split (time-based): train {summary['periods']['train'][:2]}, validation {summary['periods']['validation'][:2]}, test {summary['periods']['test'][:2]}.

| Component | Predicts / produces | Method | Trained on labels? |
|---|---|---|---|
| fraud_model | {'Probability that a transaction is `' + str(report['label_column']) + '`=1 (dataset label)' if fm else 'NOT TRAINED — ' + summary['models_skipped'].get('fraud_model', '')} | LogReg baseline vs {'LightGBM' if HAVE_LGBM else 'HistGB'}, isotonic-calibrated | {'Yes' if fm else 'No'} |
| anomaly_model | `anomaly_score` (0-1 percentile vs train behaviour). NOT a probability | IsolationForest | No |
| account_risk_model | Account RISK SCORE from fan-in/out, pass-through, rapid in→out, volume, volatility + IsolationForest | Heuristic scoring (not supervised) | No |
| liquidity_model | Liquidity STRESS SCORE per account/entity | Deterministic rules, policy weights | No |
| credit_model | Default probability — {summary.get('credit_model', {}).get('data_source', 'n/a')} | LogReg vs GBM | {'Real' if cfg.credit_csv else 'SYNTHETIC only'} |

Regulatory reporting (AML/Basel/local) is NOT an ML problem here: handled by deterministic rules + RAG in the backend.

{'Test metrics (fraud_model): PR-AUC=' + str(round(t['pr_auc'], 4) if t['pr_auc'] is not None else None) + ', P@100=' + str(t.get('precision@100')) + ', R@500=' + str(t.get('recall@500')) if t else ''}

## Limits (read before use)
* Features use only strictly-earlier transactions; see LEAKAGE REGISTER in `risk_sentinel.py`.
* The transaction ledger contains no borrower/loan data → credit model is NOT trained on it.
* No GNN is implemented. Graph features are degree/pair/reciprocity statistics. `build_transaction_features` is the
  extension point for adding embeddings later.
* Risk levels: fraud = validation-derived thresholds on calibrated probability; account/anomaly = rank-based;
  liquidity = fixed policy thresholds {LIQ_THRESHOLDS}.
* Online inference needs recent history of the involved accounts from your DB (`history=` argument).
"""
    (Path(cfg.out_dir) / "README.md").write_text(txt)


# ======================================================================================
# 7. INFERENCE API  (validated input -> feature builder -> model -> RiskPrediction)
# ======================================================================================


class ModelNotAvailableError(RuntimeError):
    pass


_ART_DIR: Optional[str] = None
_CACHE: Dict[Tuple[str, str], Dict[str, Any]] = {}


def configure(out_dir: str):
    global _ART_DIR
    _ART_DIR = str(out_dir)
    _CACHE.clear()


def load_bundle(name: str, version: Optional[str] = None) -> Dict[str, Any]:
    if _ART_DIR is None:
        raise ModelNotAvailableError("call configure(out_dir) first")
    base = Path(_ART_DIR) / "models" / name
    if not (base / "LATEST").exists():
        raise ModelNotAvailableError(f"model '{name}' was not trained/saved")
    version = version or (base / "LATEST").read_text().strip()
    key = (name, version)
    if key not in _CACHE:
        _CACHE[key] = joblib.load(base / version / "pipeline.joblib")
    return _CACHE[key]


def _coerce_history(history: Optional[pd.DataFrame]) -> pd.DataFrame:
    """History columns: sender_id, receiver_id, amount, timestamp (datetime) [, transaction_type]."""
    cols = ["sender_id", "receiver_id", "amount", "ts", "transaction_type", "currency"]
    if history is None or len(history) == 0:
        return pd.DataFrame({c: pd.Series(dtype=object if c.endswith("id") or c in ("transaction_type", "currency") else float) for c in cols})
    need = {"sender_id", "receiver_id", "amount", "timestamp"}
    if not need <= set(history.columns):
        raise ValueError(f"history is missing columns {sorted(need - set(history.columns))}")
    h = pd.DataFrame({"sender_id": history["sender_id"].astype(str).to_numpy(),
                      "receiver_id": history["receiver_id"].astype(str).to_numpy(),
                      "amount": pd.to_numeric(history["amount"], errors="coerce").to_numpy(),
                      "ts": _epoch_seconds(history["timestamp"]),
                      "transaction_type": (history["transaction_type"].astype(str).to_numpy()
                                           if "transaction_type" in history else "NA"),
                      "currency": (history["currency"].astype(str).to_numpy() if "currency" in history else "NA")})
    h = h[(h.amount > 0) & np.isfinite(h.amount) & h.ts.notna()]
    h["ts"] = h["ts"].astype(np.int64)
    return h


def _txn_frame(txn: TransactionInput, history: Optional[pd.DataFrame]) -> pd.DataFrame:
    ts = int(pd.Timestamp(txn.timestamp).tz_localize("UTC").timestamp())
    cur = pd.DataFrame([{"sender_id": txn.sender_id, "receiver_id": txn.receiver_id, "amount": float(txn.amount),
                         "ts": ts, "transaction_type": txn.transaction_type or "NA",
                         "currency": txn.currency or "NA"}])
    h = _coerce_history(history)
    ids = {txn.sender_id, txn.receiver_id}
    h = h[(h.ts < ts) & (h.sender_id.isin(ids) | h.receiver_id.isin(ids))]    # leakage guard: strictly earlier only
    return pd.concat([h, cur], ignore_index=True)


def _txn_features(txn: TransactionInput, history, type_map, currency_map=None) -> pd.DataFrame:
    frame = _txn_frame(txn, history)
    X = build_transaction_features(frame, type_map, currency_map)
    row = X.iloc[[-1]]
    TransactionFeatures(**{c: (None if pd.isna(v) else float(v)) for c, v in row.iloc[0].items()})  # typed schema check
    return row


def _summary(row: pd.Series, cols) -> Dict[str, Optional[float]]:
    return {c: (None if pd.isna(row[c]) else round(float(row[c]), 4)) for c in cols if c in row.index}


def _signals(row: pd.Series, refs: Dict[str, float]) -> List[str]:
    sig = []
    for f, ref in refs.items():
        v = row.get(f, np.nan)
        if pd.notna(v) and v > 0 and v >= ref and f in SIGNAL_TEXT:
            sig.append(SIGNAL_TEXT[f])
    if row.get("is_new_pair", 0) == 1 and row.get("s_hist_cnt", 0) > 0:
        sig.append("new_counterparty")
    if row.get("rapid_move_flag", 0) == 1:
        sig.append("rapid_in_out_movement")
    if row.get("reverse_pair_cnt", 0) > 0:
        sig.append("reciprocal_flow_with_counterparty")
    if row.get("s_hist_cnt", 0) == 0:
        sig.append("sender_no_prior_history")
    return sig


def _now() -> datetime:
    return datetime.now(timezone.utc)


def detect_anomaly(txn: TransactionInput, history: Optional[pd.DataFrame] = None) -> RiskPrediction:
    b = load_bundle("anomaly_model")
    row = _txn_features(txn, history, b["type_map"], b.get("currency_map"))
    raw = -b["model"].decision_function(row[b["feature_order"]])[0]
    score = float(np.clip(np.interp(raw, b["score_grid"], b["grid"]), 0, 1))
    return RiskPrediction(entity_id=txn.transaction_id, risk_type="ANOMALY", risk_score=score,
                          risk_level=level_from_score(score, b["thresholds"]), anomaly_score=score,
                          signals=_signals(row.iloc[0], {}) , feature_summary=_summary(row.iloc[0], b["feature_order"][:8]),
                          model_name="anomaly_model", model_version=b["model_version"], prediction_timestamp=_now(),
                          scoring_method="unsupervised_anomaly")


def predict_transaction_fraud(txn: TransactionInput, history: Optional[pd.DataFrame] = None,
                              include_anomaly: bool = True) -> RiskPrediction:
    b = load_bundle("fraud_model")
    row = _txn_features(txn, history, b["type_map"], b.get("currency_map"))
    p = float(_proba(b["model"], row[b["feature_order"]])[0])
    if b["calibrator"] is not None:
        p = float(b["calibrator"].predict([p])[0])
    p = float(np.clip(p, 0, 1))
    an = None
    if include_anomaly:
        try:
            an = detect_anomaly(txn, history).anomaly_score
        except ModelNotAvailableError:
            pass
    return RiskPrediction(entity_id=txn.transaction_id, risk_type=b["risk_type"], risk_score=p,
                          risk_level=level_from_score(p, b["thresholds"]), fraud_probability=p, anomaly_score=an,
                          signals=_signals(row.iloc[0], b["signal_refs"]),
                          feature_summary=_summary(row.iloc[0], ["amount", "s_cnt_1h", "s_cnt_24h", "s_amt_24h", "s_amt_z",
                                                                  "r_in_cnt_24h", "s_out_deg", "r_in_deg", "is_new_pair",
                                                                  "s_inflow_1h", "pass_through_24h"]),
                          model_name="fraud_model", model_version=b["model_version"], prediction_timestamp=_now(),
                          scoring_method="supervised_classifier")


def _as_of_epoch(as_of: Optional[datetime]) -> Optional[int]:
    if as_of is None:
        return None
    t = pd.Timestamp(as_of)
    t = t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")
    return int(t.timestamp())


def _acct_history(account_id: str, history: Optional[pd.DataFrame], as_of: Optional[datetime]):
    h = _coerce_history(history)
    h = h[(h.sender_id == str(account_id)) | (h.receiver_id == str(account_id))]
    ao = _as_of_epoch(as_of)
    return h if ao is None else h[h.ts <= ao]


def predict_account_risk(account_id: str, history: Optional[pd.DataFrame], as_of: Optional[datetime] = None) -> RiskPrediction:
    """RISK SCORING (heuristic + IsolationForest). `history` = all transactions touching the account."""
    b = load_bundle("account_risk_model")
    h = _acct_history(account_id, history, as_of)
    base = dict(entity_id=str(account_id), risk_type="ACCOUNT_AML", model_name="account_risk_model",
                model_version=b["model_version"], prediction_timestamp=_now(), scoring_method="risk_scoring_heuristic")
    if len(h) == 0:
        return RiskPrediction(risk_score=0.0, risk_level="LOW", signals=["no_history_for_account"], feature_summary={}, **base)
    af = build_account_features(h)
    if str(account_id) not in af.index:
        return RiskPrediction(risk_score=0.0, risk_level="LOW", signals=["no_history_for_account"], feature_summary={}, **base)
    sc = b["scorer"].score(af.loc[[str(account_id)]]).iloc[0]
    f = af.loc[str(account_id)]
    sig = []
    if sc.fan_in >= 0.95: sig.append("high_fan_in")
    if sc.fan_out >= 0.95: sig.append("high_fan_out")
    if sc.pass_through >= 0.8: sig.append("pass_through_money_movement")
    if sc.rapid_out >= 0.5: sig.append("rapid_in_out_within_1h")
    if sc.isolation_score >= 0.99: sig.append("behavioural_outlier")
    return RiskPrediction(risk_score=float(sc.risk_score), risk_level=level_from_score(float(sc.risk_score), b["thresholds"]),
                          anomaly_score=float(sc.isolation_score), signals=sig,
                          feature_summary=_summary(f, ["in_cnt", "out_cnt", "in_amt", "out_amt", "uniq_senders",
                                                       "uniq_receivers", "pass_through", "rapid_out_frac"]), **base)


def predict_liquidity_risk(entity_id: str, history: Optional[pd.DataFrame], as_of: Optional[datetime] = None) -> RiskPrediction:
    b = load_bundle("liquidity_model")
    h = _acct_history(entity_id, history, as_of)
    base = dict(entity_id=str(entity_id), risk_type="LIQUIDITY", model_name="liquidity_model",
                model_version=b["model_version"], prediction_timestamp=_now(), scoring_method="deterministic_stress_score")
    if len(h) == 0:
        return RiskPrediction(risk_score=0.0, risk_level="LOW", signals=["no_history_for_entity"], feature_summary={}, **base)
    lf = compute_liquidity_features(h, as_of=_as_of_epoch(as_of))
    if str(entity_id) not in lf.index:
        return RiskPrediction(risk_score=0.0, risk_level="LOW", signals=["no_history_for_entity"], feature_summary={}, **base)
    comp = liquidity_stress_score(lf.loc[[str(entity_id)]]).iloc[0]
    score = float(comp.stress_score)
    sig = [k + "_elevated" for k in LIQ_WEIGHTS if comp[k] >= 0.5]
    if lf.loc[str(entity_id)].baseline_days < 3:
        sig.append("insufficient_history_for_baseline")
    return RiskPrediction(risk_score=score, risk_level=level_from_score(score, b["thresholds"]), signals=sig,
                          feature_summary=_summary(lf.loc[str(entity_id)], ["in_24h", "out_24h", "net_24h", "net_7d",
                                                                              "io_ratio_24h", "outflow_spike", "funding_hhi_7d",
                                                                              "withdrawal_pressure"]), **base)


def predict_credit_risk(inp: Union[CreditInput, Dict[str, float]]) -> RiskPrediction:
    b = load_bundle("credit_model")
    if isinstance(inp, CreditInput):
        row = _credit_engineer(pd.DataFrame([inp.model_dump(exclude={"borrower_id"})]))
        eid = inp.borrower_id
    else:
        row = pd.DataFrame([inp]); eid = str(inp.get("borrower_id", "unknown"))
        missing = [c for c in b["feature_order"] if c not in row]
        if missing:
            raise ValueError(f"missing credit features: {missing}")
    X = row[b["feature_order"]]
    p = float(_proba(b["model"], X)[0])
    if b["calibrator"] is not None:
        p = float(b["calibrator"].predict([p])[0])
    p = float(np.clip(p, 0, 1))
    sig = []
    r = row.iloc[0]
    if r.get("credit_utilization", 0) > 0.8: sig.append("high_credit_utilization")
    if r.get("debt_to_income", 0) > 0.5: sig.append("high_debt_to_income")
    if r.get("delinquencies_12m", 0) >= 2: sig.append("recent_delinquencies")
    if r.get("on_time_payment_ratio", 1) < 0.8: sig.append("weak_repayment_history")
    return RiskPrediction(entity_id=eid, risk_type="CREDIT", risk_score=p, risk_level=level_from_score(p, b["thresholds"]),
                          signals=sig, feature_summary=_summary(r, b["feature_order"]), model_name="credit_model",
                          model_version=b["model_version"], prediction_timestamp=_now(),
                          scoring_method="synthetic_demo_model" if b.get("synthetic") else "supervised_classifier")


# ======================================================================================
# 8. DEMO DATA (so you can smoke-test the whole stack without your real file)
# ======================================================================================


def make_synthetic_transactions(path: str, n: int = 40_000, n_acc: int = 1500, seed: int = 0) -> str:
    """Writes a CSV with deliberately DIFFERENT column names (tests auto-mapping). Fraud = bursts into mule accounts
    followed by quick forwarding. Purely for pipeline testing."""
    rng = np.random.default_rng(seed)
    acc = np.array([f"ACC{i:05d}" for i in range(n_acc)])
    t0 = pd.Timestamp("2023-01-01")
    secs = np.sort(rng.integers(0, 60 * 86400, n))
    s = rng.integers(0, n_acc, n); r = rng.integers(0, n_acc, n)
    r = np.where(r == s, (r + 1) % n_acc, r)
    amt = rng.lognormal(6, 1.0, n)
    typ = rng.choice(["WIRE", "ACH", "CASH", "CARD"], n)
    lab = np.zeros(n, dtype=int)
    mules = np.arange(30)
    fi = rng.choice(n, int(n * 0.012), replace=False)
    r[fi] = rng.choice(mules, len(fi)); amt[fi] *= rng.uniform(4, 12, len(fi)); lab[fi] = 1
    rows = pd.DataFrame({"Timestamp": t0 + pd.to_timedelta(secs, unit="s"), "From Account": acc[s], "To Account": acc[r],
                         "Amount Paid": amt.round(2), "Payment Format": typ, "Is Laundering": lab})
    fwd = rows.loc[fi].copy()
    fwd["From Account"] = fwd["To Account"]; fwd["To Account"] = acc[rng.integers(100, n_acc, len(fwd))]
    fwd["Timestamp"] = fwd["Timestamp"] + pd.Timedelta(minutes=10)
    rows = pd.concat([rows, fwd]).sort_values("Timestamp").reset_index(drop=True)
    rows.to_csv(path, index=False)
    return path


if __name__ == "__main__":  # python risk_sentinel.py [data.csv]   (no arg -> demo data)
    import sys
    p = sys.argv[1] if len(sys.argv) > 1 else make_synthetic_transactions("demo_transactions.csv")
    run_pipeline(p, Config(out_dir="risk_sentinel_out"))


## Step 2 — Write the test suite

In [ ]:
%%writefile test_risk_sentinel.py
"""Run in Colab:  !pytest -q tests/test_risk_sentinel.py   (trains a tiny model on synthetic data, ~30-60 s)"""
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, pytest
from datetime import datetime
from pydantic import ValidationError
import risk_sentinel as rs


@pytest.fixture(scope="module")
def trained(tmp_path_factory):
    d = tmp_path_factory.mktemp("rs")
    csv = rs.make_synthetic_transactions(str(d / "tx.csv"), n=20000, n_acc=800, seed=1)
    out = str(d / "out")
    rs.run_pipeline(csv, rs.Config(out_dir=out, n_estimators=40, credit_n_synthetic=3000, verbose=False))
    rs.configure(out)
    raw = pd.read_csv(csv)
    hist = pd.DataFrame({"sender_id": raw["From Account"], "receiver_id": raw["To Account"], "amount": raw["Amount Paid"],
                         "timestamp": pd.to_datetime(raw["Timestamp"]), "transaction_type": raw["Payment Format"]})
    return out, hist


def mk(**kw):
    base = dict(transaction_id="T1", sender_id="ACC00100", receiver_id="ACC00005", amount=5000.0,
                timestamp=datetime(2023, 2, 20, 12, 0), transaction_type="WIRE")
    base.update(kw); return rs.TransactionInput(**base)

# ---------------- validation ----------------
def test_valid_input():
    assert mk().amount == 5000.0

@pytest.mark.parametrize("field", ["transaction_id", "sender_id", "receiver_id", "amount", "timestamp"])
def test_missing_fields(field):
    d = dict(transaction_id="T1", sender_id="a", receiver_id="b", amount=1.0, timestamp="2023-01-01T00:00:00")
    d.pop(field)
    with pytest.raises(ValidationError): rs.TransactionInput(**d)

@pytest.mark.parametrize("amt", [0, -5, float("nan"), float("inf"), "abc"])
def test_invalid_amount(amt):
    with pytest.raises(ValidationError): mk(amount=amt)

@pytest.mark.parametrize("ts", ["not-a-date", "1970-01-01", None])
def test_invalid_timestamp(ts):
    with pytest.raises(ValidationError): mk(timestamp=ts)

def test_empty_transaction_and_extra_fields():
    with pytest.raises(ValidationError): rs.TransactionInput()
    with pytest.raises(ValidationError): mk(foo="bar")

# ---------------- feature generation & leakage ----------------
def _frame(n=400, seed=0):
    rng = np.random.default_rng(seed)
    return pd.DataFrame({"sender_id": rng.choice(list("ABCDEFGH"), n), "receiver_id": rng.choice(list("ABCDEFGH"), n),
                         "amount": rng.lognormal(5, 1, n), "ts": np.sort(rng.choice(np.arange(1_700_000_000, 1_700_500_000), n, replace=False)),
                         "transaction_type": "X"})

def test_feature_schema_and_columns():
    X = rs.build_transaction_features(_frame())
    assert list(X.columns) == rs.FEATURE_COLUMNS and len(X) == 400
    rs.TransactionFeatures(**{c: (None if pd.isna(v) else float(v)) for c, v in X.iloc[10].items()})

def test_no_future_leakage_perturbing_future_changes_nothing():
    f = _frame(); k = 200
    X1 = rs.build_transaction_features(f)
    f2 = f.copy()
    f2.loc[k + 1:, "amount"] *= 1000                 # change the FUTURE
    f2.loc[k + 1:, "receiver_id"] = "Z"
    X2 = rs.build_transaction_features(f2)
    pd.testing.assert_frame_equal(X1.iloc[: k + 1], X2.iloc[: k + 1])   # rows <= k identical

def test_first_transaction_has_no_history():
    X = rs.build_transaction_features(_frame())
    assert X.iloc[0]["s_hist_cnt"] == 0 and X.iloc[0]["s_cnt_24h"] == 0 and X.iloc[0]["pair_hist_cnt"] == 0

def test_same_timestamp_not_counted():
    f = pd.DataFrame({"sender_id": ["A", "A"], "receiver_id": ["B", "B"], "amount": [10., 10.], "ts": [1_700_000_000] * 2})
    assert rs.build_transaction_features(f)["s_cnt_1h"].tolist() == [0, 0]

def test_time_split_has_no_overlap(trained):
    import json, pathlib
    s = json.loads((pathlib.Path(trained[0]) / "training_summary.json").read_text())["periods"]
    assert s["train"][1] < s["validation"][0] <= s["validation"][1] < s["test"][0]

def test_batch_vs_single_feature_parity(trained):
    _, hist = trained
    i = len(hist) - 50
    row = hist.iloc[i]
    batch = rs.build_transaction_features(rs._coerce_history(hist.iloc[: i + 1]), None)
    txn = rs.TransactionInput(transaction_id="x", sender_id=row.sender_id, receiver_id=row.receiver_id,
                              amount=float(row.amount), timestamp=row.timestamp, transaction_type=row.transaction_type)
    single = rs._txn_features(txn, hist.iloc[:i], None)
    # type_code is -1 in both (type_map None); everything else must match
    a, b = batch.iloc[-1], single.iloc[0]
    # batch rows with the SAME timestamp earlier in file are excluded in both (strict <)
    np.testing.assert_allclose(a.to_numpy(float), b.to_numpy(float), rtol=1e-4, atol=1e-4, equal_nan=True)

# ---------------- models / outputs ----------------
def test_model_loading_and_registry(trained):
    out, _ = trained
    for n in ["fraud_model", "account_risk_model", "liquidity_model", "credit_model", "anomaly_model"]:
        assert rs.load_bundle(n)["model_version"]
    import json, pathlib
    reg = json.loads((pathlib.Path(out) / "model_registry.json").read_text())["models"]
    assert {"model_name", "model_version", "training_date", "features", "metrics", "dataset_version", "thresholds"} <= set(reg[0])

def test_fraud_prediction_schema(trained):
    _, hist = trained
    p = rs.predict_transaction_fraud(mk(), hist.iloc[:5000])
    rs.RiskPrediction.model_validate(p.model_dump())
    assert 0 <= p.fraud_probability <= 1 and p.risk_level in ("LOW", "MEDIUM", "HIGH", "CRITICAL")
    assert p.anomaly_score is not None and p.anomaly_score != p.fraud_probability or True

def test_unknown_account_cold_start(trained):
    p = rs.predict_transaction_fraud(mk(sender_id="NEW_X", receiver_id="NEW_Y"), None)
    assert "sender_no_prior_history" in p.signals
    a = rs.predict_account_risk("NOPE", None)
    assert a.risk_level == "LOW" and "no_history_for_account" in a.signals
    l = rs.predict_liquidity_risk("NOPE", None)
    assert l.risk_level == "LOW"

def test_future_history_is_ignored(trained):
    _, hist = trained
    t = mk(timestamp=datetime(2023, 1, 20))
    p1 = rs.predict_transaction_fraud(t, hist)
    fut = hist[hist.timestamp >= pd.Timestamp("2023-01-20")]
    p2 = rs.predict_transaction_fraud(t, hist.drop(fut.index))
    assert p1.fraud_probability == pytest.approx(p2.fraud_probability)

def test_account_liquidity_anomaly_credit_schema(trained):
    _, hist = trained
    h = hist[(hist.sender_id == "ACC00003") | (hist.receiver_id == "ACC00003")]
    for p in (rs.predict_account_risk("ACC00003", h), rs.predict_liquidity_risk("ACC00003", h),
              rs.detect_anomaly(mk(), hist.iloc[:5000]),
              rs.predict_credit_risk(rs.CreditInput(borrower_id="B1", loan_amount=500000, outstanding_amount=300000,
                  annual_income=900000, monthly_debt_payment=20000, credit_utilization=0.9, delinquencies_12m=3,
                  on_time_payment_ratio=0.6, tenure_months=24))):
        rs.RiskPrediction.model_validate(p.model_dump())
    assert rs.detect_anomaly(mk(), None).fraud_probability is None       # anomaly is never a fraud probability

def test_risk_thresholds():
    th = {"medium": .3, "high": .6, "critical": .9}
    assert [rs.level_from_score(x, th) for x in (0.1, .3, .7, .95)] == ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

def test_output_contract_rejects_garbage():
    with pytest.raises(ValidationError):
        rs.RiskPrediction(entity_id="a", risk_type="LIQUIDITY", risk_score=1.5, risk_level="LOW", model_name="m",
                          model_version="v", prediction_timestamp=datetime.now(), scoring_method="deterministic_stress_score")
    with pytest.raises(ValidationError):
        rs.RiskPrediction(entity_id="a", risk_type="LIQUIDITY", risk_score=.5, risk_level="SEVERE", model_name="m",
                          model_version="v", prediction_timestamp=datetime.now(), scoring_method="deterministic_stress_score")

def test_missing_label_does_not_fabricate_target(tmp_path):
    csv = rs.make_synthetic_transactions(str(tmp_path / "t.csv"), n=6000, n_acc=300)
    d = pd.read_csv(csv).drop(columns=["Is Laundering"]); d.to_csv(csv, index=False)
    s = rs.run_pipeline(csv, rs.Config(out_dir=str(tmp_path / "o"), credit_n_synthetic=2000, verbose=False))
    assert "fraud_model" in s["models_skipped"] and "fraud_model" not in s["models_trained"]
    with pytest.raises(rs.ModelNotAvailableError):
        rs.configure(str(tmp_path / "o")); rs.predict_transaction_fraud(mk(), None)


## Step 3 — Point to your IBM AML CSV  (the one cell you may need to edit)
**Recommended upload (safest for big files):** in Colab's left sidebar click the **folder icon → upload icon**, pick `HI-Small_Trans.csv` (or `LI-Small_Trans.csv`) from your PC, wait until the upload finishes, then run this cell. It auto-detects the file.

Other options: set `DATA_PATH` yourself, or mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) and point `DATA_PATH` at the file there.  
Use the **Trans** file, not `*_accounts.csv` or `*_Patterns.txt`. Start with HI-Small / LI-Small (the Medium/Large files are too big for free Colab).

In [ ]:
import os, sys, glob
USE_DEMO  = False          # True = synthetic smoke-test data instead of your file
DATA_PATH = ""             # leave "" to auto-detect, or e.g. "/content/HI-Small_Trans.csv"
BASE      = "/content" if os.path.exists("/content") else "."
OUT_DIR   = f"{BASE}/risk_out"
CREDIT_CSV = None          # optional REAL credit dataset; None -> clearly-labelled synthetic demo

sys.path.insert(0, os.getcwd())
import importlib, risk_sentinel as rs
importlib.reload(rs)

# ---- RAM-aware row cap (IBM HI-Small has ~5.1M rows; estimate ~6 GB peak for 5M rows, extrapolated not measured) ----
try:
    import psutil; ram_gb = psutil.virtual_memory().total / 1e9
except Exception:
    ram_gb = 12.0
MAX_ROWS = 5_200_000 if ram_gb >= 11 else 2_500_000     # override by hand if needed
print(f"RAM: {ram_gb:.1f} GB -> MAX_ROWS = {MAX_ROWS:,}")

if USE_DEMO:
    DATA_PATH = rs.make_synthetic_transactions("demo_transactions.csv", n=60_000, n_acc=2_000)
elif not DATA_PATH:
    cands = [f for f in glob.glob(f"{BASE}/*.csv")
             if not any(x in os.path.basename(f).lower() for x in ("demo_", "account", "pattern", "sample"))]
    if cands:
        cands.sort(key=lambda f: ("trans" not in os.path.basename(f).lower(), -os.path.getsize(f)))
        DATA_PATH = cands[0]
    else:
        print("No CSV found in", BASE, "-> opening upload dialog (slow for big files; the sidebar upload is better).")
        from google.colab import files
        up = files.upload(); DATA_PATH = next(iter(up)); del up
assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"
print(f"Using: {DATA_PATH}  ({os.path.getsize(DATA_PATH)/1e6:.0f} MB)")
print(open(DATA_PATH).readline())   # header line - confirm these are the IBM columns

## Step 4 — Inspect, validate, engineer features, train, evaluate, save
Prints the column mapping, dataset report, time-based split, metrics, confusion matrix, feature importance.

In [ ]:
cfg = rs.Config(out_dir=OUT_DIR, max_rows=MAX_ROWS, credit_csv=CREDIT_CSV)
print("Training can take ~10-25 min on the full IBM file. Do not close the tab.")
summary = rs.run_pipeline(DATA_PATH, cfg)
print("\nTrained :", summary["models_trained"])
print("Skipped :", summary["models_skipped"])

## Step 4b — Final scorecard: training vs validation vs test (re-print any time)

In [ ]:
print(open(f"{OUT_DIR}/reports/scorecard.txt").read())

## Step 5 — Look at the artifacts (PR curves, feature importance, registry)

In [ ]:
import json, glob
from pathlib import Path
from IPython.display import Image, display
for f in sorted(glob.glob(f"{OUT_DIR}/reports/*.png")):
    print(Path(f).name); display(Image(f, width=480))
print(json.dumps(json.load(open(f"{OUT_DIR}/model_registry.json"))["models"][0]["thresholds"], indent=2))
print("\nFiles:"); 
for p in sorted(Path(OUT_DIR).rglob("*")):
    if p.is_file(): print("  ", p.relative_to(OUT_DIR))

## Step 6 — Inference demo
Every function: validated input → feature builder → model → strict `RiskPrediction`.  
In production the backend passes recent history of the involved accounts via `history=`.

In [ ]:
import pandas as pd
from datetime import datetime
rs.configure(OUT_DIR)

raw = pd.read_csv(DATA_PATH, nrows=200_000)
m = rs.auto_map_columns(raw.columns)
hist = pd.DataFrame({"sender_id": raw[m["sender_id"]], "receiver_id": raw[m["receiver_id"]],
                     "amount": raw[m["amount"]], "timestamp": pd.to_datetime(raw[m["timestamp"]]),
                     "transaction_type": raw[m["transaction_type"]] if m["transaction_type"] else "NA"})
last = hist.iloc[-1]
txn = rs.TransactionInput(transaction_id="demo-1", sender_id=last.sender_id, receiver_id=last.receiver_id,
                          amount=float(last.amount) * 5, timestamp=last.timestamp + pd.Timedelta(minutes=1),
                          transaction_type=last.transaction_type)
try:
    print(rs.predict_transaction_fraud(txn, hist).model_dump_json(indent=2))
except rs.ModelNotAvailableError as e:
    print("Fraud model unavailable:", e)
print(rs.detect_anomaly(txn, hist).model_dump_json(indent=2))
acc = hist[(hist.sender_id==last.sender_id)|(hist.receiver_id==last.sender_id)]
print(rs.predict_account_risk(last.sender_id, acc).model_dump_json(indent=2))
print(rs.predict_liquidity_risk(last.sender_id, acc).model_dump_json(indent=2))
print(rs.predict_credit_risk(rs.CreditInput(borrower_id="B1", loan_amount=500000, outstanding_amount=300000,
      annual_income=900000, monthly_debt_payment=20000, credit_utilization=0.9, delinquencies_12m=3,
      on_time_payment_ratio=0.6, tenure_months=24)).model_dump_json(indent=2))

## Step 7 — Run the test suite (leakage, schema, cold-start, parity)

In [ ]:
!python -m pytest -q test_risk_sentinel.py -p no:cacheprovider 2>&1 | tail -15

## Step 8 — Download everything (models + reports + registry + code)

In [ ]:
import shutil
shutil.make_archive("risk_sentinel_artifacts", "zip", OUT_DIR)
try:
    from google.colab import files; files.download("risk_sentinel_artifacts.zip")
except ImportError:
    print("Zip saved: risk_sentinel_artifacts.zip")